# 💳 Project Task: GoPay Fintech Analytics
## Data Cleaning, Feature Engineering & Exploratory Data Analysis

**Module 2 — Python for Data Analysis | Purwadhika Digital Technology School**

---

> ⚠️ **Jangan di-run dulu.** Copy notebook ini terlebih dahulu, baru kerjakan di file copy-an kamu.

---

### Konteks Bisnis
GoPay telah berkembang menjadi tulang punggung ekosistem Super App GoTo. Fitur GoPayLater — layanan kredit berbasis limit — berhasil mendongkrak GTV, namun kini menghadapi masalah serius: tingkat NPL (Non-Performing Loan) yang meningkat, bug validasi limit kredit, dan inkonsistensi data dari puluhan micro-service.

Kamu berperan sebagai Data Analyst di tim **Risk Management GoPay** yang diminta untuk membersihkan data, mengidentifikasi pola gagal bayar, dan memberikan rekomendasi perbaikan credit scoring.

**Dataset (3 tabel):**
- `gopay_users.csv` — 35.000 baris (Dimensi User)
- `gopay_services.csv` — 20 baris (Dimensi Layanan)
- `gopay_transactions.csv` — 300.000 baris (Fakta Transaksi)

---

### ⚠️ Catatan Penting
- Task ini **open-ended** — tidak ada satu jawaban yang mutlak benar
- Yang dinilai: **ketepatan keputusan**, **kualitas justifikasi**, dan **kedalaman analisis**
- Setiap keputusan di Data Cleaning & Feature Engineering **wajib disertai penjelasan** di markdown cell
- EDA dikerjakan **tanpa visualisasi** — gunakan pandas aggregation, filtering, sorting, dan merge

---

### 🚨 Business Context Error (Wajib Diinvestigasi)
> Lebih dari **50% transaksi GoPayLater** memiliki `amount` yang **melebihi `paylater_limit`** user yang bersangkutan.  
> Ini adalah bug sistematis pada validasi limit kredit — bukan sekadar outlier biasa.  
> Identifikasi, kuantifikasi dampak finansialnya, dan rekomendasikan perbaikan.

---
## 0. Import & Load Data

In [4]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [5]:
# Load semua dataset dari folder dataset
# Find gopay_users.csv in dataset or its subfolders
from pathlib import Path

# Cari file CSV dari folder notebook saat ini dan semua subfoldernya
data_dir = Path.cwd()

def find_csv(filename):
    matches = list(data_dir.rglob(filename))
    
    if not matches:
        raise FileNotFoundError(
            f"File '{filename}' tidak ditemukan di folder: {data_dir.resolve()}"
        )
    
    return matches[0]

# Menentukan lokasi file
users_path = find_csv('gopay_users.csv')
services_path = find_csv('gopay_services.csv')
transactions_path = find_csv('gopay_transactions.csv')

# Load dataset
df_users_raw = pd.read_csv(users_path)
df_services_raw = pd.read_csv(services_path)
df_trx_raw = pd.read_csv(transactions_path)

# Membuat salinan kerja agar data mentah tetap tersimpan
df_users = df_users_raw.copy()
df_services = df_services_raw.copy()
df_trx = df_trx_raw.copy()

print("Dataset berhasil dimuat:")
print(f"- Users        : {df_users.shape} | {users_path}")
print(f"- Services     : {df_services.shape} | {services_path}")
print(f"- Transactions : {df_trx.shape} | {transactions_path}")


Dataset berhasil dimuat:
- Users        : (35000, 5) | /Users/nazmaaulia/Downloads/gopaylater_DataAnalysis/gopay_users.csv
- Services     : (20, 3) | /Users/nazmaaulia/Downloads/gopaylater_DataAnalysis/gopay_services.csv
- Transactions : (300000, 8) | /Users/nazmaaulia/Downloads/gopaylater_DataAnalysis/gopay_transactions.csv


---
## 2. Data Cleaning

### 2.1 Eksplorasi Awal (Wajib)

Lakukan eksplorasi menyeluruh pada **ketiga tabel** sebelum membersihkan data apapun.

In [6]:
# Shape dan info umum — lakukan untuk ketiga tabel

display(df_users_raw,
df_services_raw,
df_trx_raw)

df_users = df_users_raw.copy()
df_services = df_services_raw.copy()
df_trx = df_trx_raw.copy()

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
0,GP-000001,2022-05-26,Basic,NaN,0
1,GP-000002,2023-07-12,Plus,751.00,500000
2,GP-000003,2022-05-09,Plus,728.00,1500000
3,GP-000004,2021-08-28,Basic,394.00,0
4,GP-000005,2021-05-31,Basic,333.00,0
...,...,...,...,...,...
34995,GP-034996,2022-03-12,Plus,544.00,1500000
34996,GP-034997,2023-07-23,Plus,NaN,5000000
34997,GP-034998,2023-08-06,Plus,665.00,1500000
34998,GP-034999,2022-05-29,Basic,391.00,0


,service_id,service_name,category
0,SVC-001,GoRide,Mobility
1,SVC-002,GoCar,Mobility
2,SVC-003,GoBluebird,Mobility
3,SVC-004,GoFood,Food Delivery
4,SVC-005,GoMart,Food Delivery
5,SVC-006,GoSend,Logistics
6,SVC-007,GoBox,Logistics
7,SVC-008,Pulsa/Data,Digital Goods & Bills
8,SVC-009,PLN,Digital Goods & Bills
9,SVC-010,PDAM,Digital Goods & Bills


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,gopay,34229,0.00,Paid
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,gopay,234183,0.00,Paid
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,PayLater,380431,0.00,Paid
...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,CASH,327414,0.00,Paid
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid


In [7]:
# Tipe data seluruh kolom
df_users_raw.info()
df_services_raw.info()
df_trx_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                35000 non-null  str    
 1   join_date              35000 non-null  str    
 2   gopay_tier             35000 non-null  str    
 3   internal_credit_score  29750 non-null  float64
 4   paylater_limit         35000 non-null  int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 2.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   service_id    20 non-null     str  
 1   service_name  20 non-null     str  
 2   category      20 non-null     str  
dtypes: str(3)
memory usage: 1.2 KB
<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
--- 

In [8]:
# Missing values: jumlah dan persentase per kolom, per tabel
# Tampilkan hanya kolom yang memiliki missing values
display(df_users_raw.isnull().sum())
display(df_services_raw.isnull().sum())
display(df_trx_raw.isnull().sum())

na_values = df_users.isnull().sum()
print('Total nilai kosong pada tiap kolom')
na_values.apply(lambda x : f'{x} - {x/len(df_users):.2%}')

user_id                     0
join_date                   0
gopay_tier                  0
internal_credit_score    5250
paylater_limit              0
dtype: int64

service_id      0
service_name    0
category        0
dtype: int64

trx_id            0
user_id           0
service_id        0
trx_date          0
payment_method    0
amount            0
late_fee          0
payment_status    0
dtype: int64

Total nilai kosong pada tiap kolom


user_id                      0 - 0.00%
join_date                    0 - 0.00%
gopay_tier                   0 - 0.00%
internal_credit_score    5250 - 15.00%
paylater_limit               0 - 0.00%
dtype: str

In [9]:
# Distribusi kolom-kolom kritis
# payment_method, amount, late_fee, payment_status, gopay_tier, internal_credit_score
display
(df_trx['amount'].describe(),
df_trx['payment_method'].describe(),
df_trx['late_fee'].describe(),
df_trx['payment_status'].describe()
)

(count    300000.00
 mean     447504.65
 std      490747.56
 min       15004.00
 25%      213294.75
 50%      413388.50
 75%      612431.75
 max     9965039.00
 Name: amount, dtype: float64,
 count     300000
 unique         8
 top        GoPay
 freq       90240
 Name: payment_method, dtype: object,
 count     300000.00
 mean       26734.99
 std      1591437.75
 min       -15000.00
 25%            0.00
 50%            0.00
 75%            0.00
 max     99999999.00
 Name: late_fee, dtype: float64,
 count     300000
 unique         3
 top         Paid
 freq      273921
 Name: payment_status, dtype: object)

In [10]:
display(
df_users['gopay_tier'].describe(),
df_users['internal_credit_score'].describe()
)

count     35000
unique        2
top        Plus
freq      21062
Name: gopay_tier, dtype: object

count   29750.00
mean      534.25
std       153.50
min       300.00
25%       415.00
50%       490.00
75%       662.00
max       849.00
Name: internal_credit_score, dtype: float64

In [11]:
# Cek konsistensi relasi antar tabel
# Apakah semua service_id di transactions ada di services?
# Apakah semua user_id di transactions ada di users?
display(
    df_trx[
        ~df_trx['user_id'].isin(df_users['user_id'])
    ],
    df_trx[
        ~df_trx['service_id'].isin(df_services['service_id'])
    ]
)

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status


**✍️ Ringkasan Temuan Eksplorasi:**

*(Kolom apa yang bermasalah di setiap tabel, seberapa parah, dan prioritas penanganan kamu)*

> 

---
### 2.2 Kerangka Identifikasi Missing Values

Sebelum menangani missing values pada kolom manapun, identifikasi dulu **jenis missing value-nya**.

| Jenis | Definisi Singkat | Implikasi Penanganan | Contoh di Dataset Ini |
|---|---|---|---|
| **MCAR** *(Missing Completely At Random)* | Nilai kosong tidak berkaitan dengan kolom lain. Pola missing benar-benar acak. | Relatif aman di-impute atau di-drop tanpa bias signifikan. | Sebagian kecil `internal_credit_score` kosong karena gangguan sistem scraping acak. |
| **MAR** *(Missing At Random)* | Nilai kosong berkaitan dengan kolom **lain**, bukan dengan nilai kolom itu sendiri. | Imputation berbasis kolom lain lebih tepat. Drop bisa menyebabkan bias. | `internal_credit_score` kosong mungkin berkorelasi dengan `gopay_tier` atau `join_date`. |
| **MNAR** *(Missing Not At Random)* | Nilai kosong berkaitan langsung dengan nilai yang seharusnya ada. Ada alasan sistematis. | Imputation apapun berisiko misleading. Perlu keputusan bisnis eksplisit. | `internal_credit_score` kosong justru karena user tidak pernah bertransaksi — nilai kosong itu sendiri adalah sinyal risiko. |

> 💡 **Cara Menggunakan Kerangka Ini:**
> Untuk setiap kolom bermasalah, tanyakan:
> 1. Apakah pola missing-nya acak, atau ada pola tertentu?
> 2. Apakah nilai kosong berkaitan dengan kolom lain?
> 3. Apakah nilai kosong itu sendiri mengandung informasi bisnis?
>
> Justifikasi reasoning kamu lebih penting dari labelnya.

In [12]:
display(

df_users[
    (df_users['internal_credit_score'].isnull())
    & (df_users['paylater_limit'] > 0)
    & (df_users['gopay_tier'] == 'Basic')
] # Secara Akun Basic sudah benar internal_credit_score null karena paylater_limit = 0  (benar)


,df_users[
    (df_users['paylater_limit'] > 0)
    & (df_users['gopay_tier'] == 'Basic')
] # Secara Akun Basic tidak ada yang PayLater Limit > 0  (benar)

,df_users[
    (df_users['internal_credit_score'].notnull())
    & (df_users['gopay_tier'] == 'Basic')
] # Secara Akun Basic seharusnya internal_credit_score = 0 (Masalah)

,df_users[
    (df_users['internal_credit_score'].isnull())
    & (df_users['paylater_limit'] > 0)
    & (df_users['gopay_tier'] == 'Plus')
] # Akun Plus Seharusnya memiliki credi score semua (masalah)

,df_users[
    (df_users['paylater_limit'] == 0)
    & (df_users['gopay_tier'] == 'Plus')
] #  Akun Plus semua memiliki Payment_Limit yang tidak 0 (benar)
)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
3,GP-000004,2021-08-28,Basic,394.00,0
4,GP-000005,2021-05-31,Basic,333.00,0
11,GP-000012,2022-02-05,Basic,333.00,0
12,GP-000013,2023-01-04,Basic,405.00,0
13,GP-000014,2023-02-20,Basic,339.00,0
...,...,...,...,...,...
34989,GP-034990,2021-06-08,Basic,497.00,0
34992,GP-034993,2021-11-15,Basic,493.00,0
34994,GP-034995,2022-05-05,Basic,367.00,0
34998,GP-034999,2022-05-29,Basic,391.00,0


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
7,GP-000008,2023-03-26,Plus,NaN,500000
9,GP-000010,2021-04-20,Plus,NaN,5000000
19,GP-000020,2022-06-23,Plus,NaN,3000000
25,GP-000026,2022-01-15,Plus,NaN,500000
35,GP-000036,2022-07-20,Plus,NaN,1500000
...,...,...,...,...,...
34973,GP-034974,2022-06-14,Plus,NaN,3000000
34977,GP-034978,2022-04-26,Plus,NaN,500000
34980,GP-034981,2023-02-28,Plus,NaN,1500000
34988,GP-034989,2021-03-16,Plus,NaN,1500000


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


---
### 2.3 Penanganan Kolom `payment_method`

Kolom ini memiliki 8 varian penulisan untuk 3 metode pembayaran yang berbeda, akibat inkonsistensi penamaan antar micro-service.

| Varian Asli | Metode Sebenarnya |
|---|---|
| `GoPay`, `gopay`, `GO-PAY` | GoPay (saldo digital) |
| `GoPayLater`, `PayLater`, `gopay_later` | GoPayLater (kredit) |
| `Cash`, `CASH` | Cash |

> 🧠 **Critical Thinking Prompt:**  
> Setelah standarisasi, periksa ulang: apakah ada user **Basic tier** yang menggunakan GoPayLater?  
> Secara aturan bisnis, PayLater hanya boleh digunakan oleh user Plus.  
> Jika ada, apakah itu error data atau bug sistem validasi?

In [13]:
# Lihat semua nilai unik di payment_method beserta frekuensinya
df_trx['payment_method'].unique()

<ArrowStringArray>
[      'GoPay',       'gopay',    'PayLater',  'GoPayLater',      'GO-PAY',
        'Cash', 'gopay_later',        'CASH']
Length: 8, dtype: str

**✍️ Mapping standarisasi yang kamu buat:**
- Varian asli → nilai standar (tuliskan mapping lengkapnya):
- Format standar yang kamu pilih dan alasannya:
- Temuan setelah standarisasi (apakah ada Basic tier yang pakai GoPayLater?):

> 

In [14]:
# TODO: Standarisasi payment_method
# Simpan hasil ke kolom baru: payment_method_clean
def normalize_payment_method(x):
    if x['payment_method'] in ['gopay', 'GO-PAY', 'GoPay']:
        return 'GoPay'
    elif x['payment_method'] in ['GoPayLater', 'PayLater', 'gopay_later']:
        return 'GoPayLater'
    elif x['payment_method'] in ['Cash', 'CASH']:
        return 'Cash'

df_trx_clean = df_trx.copy()

df_trx_clean['payment_method'] = df_trx_clean.apply(normalize_payment_method, axis=1)
df_trx_clean

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,GoPay,34229,0.00,Paid
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,GoPay,234183,0.00,Paid
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,GoPayLater,380431,0.00,Paid
...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,Cash,327414,0.00,Paid
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid


In [15]:
# Verifikasi: cek Basic tier yang menggunakan GoPayLater setelah standarisasi
user_trx = df_users.merge(df_trx_clean,'inner',on='user_id')
user_trx

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068,0.00,Paid
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GoPay,195808,0.00,Paid
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,GoPay,606859,0.00,Paid
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899,0.00,Paid
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928,0.00,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...
299995,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650,0.00,Paid
299996,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,GoPay,138743,0.00,Paid
299997,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400,0.00,Paid
299998,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,GoPay,568217,0.00,Paid


In [16]:
user_trx[
    (user_trx['payment_method'] =='GoPayLater')
    &(user_trx['gopay_tier'] == 'Basic')
]

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928,0.00,Paid
33,GP-000005,2021-05-31,Basic,333.00,0,GTRX-0026167,SVC-003,2023-03-07 07:00:00,GoPayLater,195639,0.00,Paid
34,GP-000005,2021-05-31,Basic,333.00,0,GTRX-0089251,SVC-014,2023-02-08 13:00:00,GoPayLater,659008,0.00,Paid
36,GP-000005,2021-05-31,Basic,333.00,0,GTRX-0230522,SVC-003,2023-08-27 10:00:00,GoPayLater,367491,0.00,Paid
87,GP-000012,2022-02-05,Basic,333.00,0,GTRX-0090932,SVC-016,2023-01-03 20:00:00,GoPayLater,154838,0.00,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...
299988,GP-034999,2022-05-29,Basic,391.00,0,GTRX-0283795,SVC-005,2023-06-03 07:00:00,GoPayLater,648804,0.00,Paid
299990,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0039481,SVC-007,2023-09-13 08:00:00,GoPayLater,643625,22789.00,Default
299991,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0050933,SVC-017,2023-01-17 16:00:00,GoPayLater,222357,10905.00,Default
299997,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400,0.00,Paid


---
### 2.4 Penanganan `internal_credit_score`

Kolom `internal_credit_score` di tabel users memiliki **5.250 nilai kosong (~15%)** — variabel kritis untuk analisis risiko kredit.

> 🧠 **Critical Thinking Prompt:**  
> Apakah nilai kosong ini karena sistem gagal mencatat, atau karena user memang belum punya histori kredit?  
> User tanpa credit score = *unscored* — di industri fintech, ini dianggap risiko tersendiri.  
> Keputusan kamu di sini akan langsung mempengaruhi hasil analisis profil risiko di Section 4.

In [17]:
# Investigasi pola missing values pada internal_credit_score
# Apakah berkorelasi dengan gopay_tier, paylater_limit, atau join_date?
display(

user_trx[
    user_trx['internal_credit_score'].isnull()
],
user_trx[
    user_trx['internal_credit_score'].notnull()
]
)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068,0.00,Paid
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GoPay,195808,0.00,Paid
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,GoPay,606859,0.00,Paid
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899,0.00,Paid
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928,0.00,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...
299961,GP-034997,2023-07-23,Plus,NaN,5000000,GTRX-0093771,SVC-003,2023-04-27 12:00:00,GoPay,230848,0.00,Paid
299962,GP-034997,2023-07-23,Plus,NaN,5000000,GTRX-0222554,SVC-009,2023-02-24 21:00:00,GoPay,294985,0.00,Paid
299963,GP-034997,2023-07-23,Plus,NaN,5000000,GTRX-0265566,SVC-002,2023-01-06 22:00:00,GoPayLater,293931,0.00,Paid
299964,GP-034997,2023-07-23,Plus,NaN,5000000,GTRX-0275258,SVC-001,2023-11-21 21:00:00,GoPay,304070,0.00,Paid


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
6,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0017474,SVC-010,2023-06-29 12:00:00,GoPay,200782,0.00,Paid
7,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0085613,SVC-007,2023-08-14 06:00:00,GoPay,251279,0.00,Paid
8,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0089307,SVC-008,2023-01-24 20:00:00,GoPay,191460,0.00,Paid
9,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0101411,SVC-003,2023-05-24 07:00:00,GoPay,88206,0.00,Paid
10,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0103354,SVC-007,2023-05-18 08:00:00,GoPayLater,357436,37815.00,Default
...,...,...,...,...,...,...,...,...,...,...,...,...
299995,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650,0.00,Paid
299996,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,GoPay,138743,0.00,Paid
299997,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400,0.00,Paid
299998,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,GoPay,568217,0.00,Paid


### Berdasarkan data perbandingan diatas mendapat kesimpulan missing value pada internal_credit_score tidak berhubungan dengan (gopay_tier, paylater_limit, atau join_date)

In [18]:
# Cek: apakah user yang credit_score-nya missing lebih banyak yang default?
# Hint: merge dengan df_trx, lalu bandingkan default rate

user_trx.loc[
    user_trx['internal_credit_score'].isnull(),
    'payment_status'
].value_counts(normalize=True) * 100


payment_status
Paid      91.33
Default    5.07
Pending    3.60
Name: proportion, dtype: float64

**✍️ Analisis & Justifikasi:**
- **Jenis missing value (MCAR / MAR / MNAR):** dan alasan klasifikasi kamu:
- Temuan investigasi pola missing (berkorelasi dengan tier? join_date?):
- Apakah user tanpa credit score memiliki default rate yang berbeda?
- Keputusan penanganan (drop / impute / pertahankan NaN) dan alasan:

> 

In [19]:
# TODO: Implementasi penanganan missing values internal_credit_score
display(
df_users[
    (df_users['gopay_tier'] == 'Basic')
    &(df_users['internal_credit_score'].isnull())
]
,df_users[
    (df_users['gopay_tier'] == 'Plus')
    &(df_users['internal_credit_score'].isnull())
]
)


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
0,GP-000001,2022-05-26,Basic,NaN,0
28,GP-000029,2023-01-14,Basic,NaN,0
33,GP-000034,2022-11-16,Basic,NaN,0
42,GP-000043,2022-10-02,Basic,NaN,0
47,GP-000048,2021-02-15,Basic,NaN,0
...,...,...,...,...,...
34949,GP-034950,2022-03-20,Basic,NaN,0
34950,GP-034951,2022-12-25,Basic,NaN,0
34953,GP-034954,2021-01-29,Basic,NaN,0
34972,GP-034973,2023-05-01,Basic,NaN,0


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
7,GP-000008,2023-03-26,Plus,NaN,500000
9,GP-000010,2021-04-20,Plus,NaN,5000000
19,GP-000020,2022-06-23,Plus,NaN,3000000
25,GP-000026,2022-01-15,Plus,NaN,500000
35,GP-000036,2022-07-20,Plus,NaN,1500000
...,...,...,...,...,...
34973,GP-034974,2022-06-14,Plus,NaN,3000000
34977,GP-034978,2022-04-26,Plus,NaN,500000
34980,GP-034981,2023-02-28,Plus,NaN,1500000
34988,GP-034989,2021-03-16,Plus,NaN,1500000


In [20]:
df_users = df_users[
    (df_users['gopay_tier'] == 'Plus')
    &(df_users['internal_credit_score'].notnull())
]

user_trx = user_trx[
    (user_trx['gopay_tier'] == 'Plus')
    &(user_trx['internal_credit_score'].notnull())
]

df_users[
    (df_users['gopay_tier'] == 'Plus')
    &(df_users['internal_credit_score'].isnull())
]



,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


Penanganan missing value internal_credit_score adalah dipertahankan 
Alasan:
- Tidak terdapat pola tertentu yang menyebabkan kekosongan credit_score
- Karena 2122 rows merupakan Tier Basic, yang masuk akal bila credit_score nya kosong.
- Sedangkan 3128 rows merupakan Tier Plus, dilakukan penghapusan

---
### 2.5 Penanganan `late_fee`

Kolom `late_fee` memiliki dua jenis anomali yang berbeda sifatnya — tangani secara terpisah.

> 🧠 **Critical Thinking Prompt:**  
> Di industri fintech, denda keterlambatan diatur oleh regulasi OJK.  
> Nilai `late_fee` yang sangat besar bisa berarti bug sistem, bukan kebijakan yang valid.  
> Keputusan kamu harus mempertimbangkan aspek **compliance**, bukan hanya statistik.

In [21]:
# Investigasi distribusi late_fee secara menyeluruh
# Berapa nilai negatif? Berapa nilai ekstrem?
df_trx_clean[
    df_trx_clean['late_fee'] <0
].sort_values('late_fee',ascending=True)


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
1586,GTRX-0001587,GP-010619,SVC-018,2023-12-02 02:00:00,GoPayLater,4196479,-15000.00,Default
192999,GTRX-0193000,GP-014505,SVC-016,2023-01-02 03:00:00,GoPayLater,1435426,-15000.00,Default
192697,GTRX-0192698,GP-028786,SVC-013,2023-05-03 11:00:00,GoPayLater,762842,-15000.00,Default
192642,GTRX-0192643,GP-032841,SVC-001,2023-10-09 02:00:00,GoPayLater,786849,-15000.00,Default
190444,GTRX-0190445,GP-026844,SVC-002,2023-10-20 15:00:00,GoPayLater,753778,-15000.00,Default
...,...,...,...,...,...,...,...,...
79492,GTRX-0079493,GP-019686,SVC-015,2023-11-13 13:00:00,GoPayLater,529118,-15000.00,Default
77798,GTRX-0077799,GP-017858,SVC-013,2023-03-15 14:00:00,GoPayLater,406025,-15000.00,Default
77677,GTRX-0077678,GP-022504,SVC-008,2023-05-06 04:00:00,GoPayLater,515446,-15000.00,Default
70689,GTRX-0070690,GP-002392,SVC-004,2023-08-19 06:00:00,GoPayLater,460722,-15000.00,Default


In [22]:
# Anomali 1: Nilai negatif
# Apakah terjadi pada payment_status tertentu?
# display(
# user_trx.loc[
#     user_trx['late_fee'] <0
# ,'payment_status'].unique()

# ,user_trx[
#     (user_trx['payment_status'] == 'Default')
#     & (user_trx['late_fee'] > 0)
# ]


# ,user_trx[
#     (user_trx['payment_status'] == 'Default')
#     & (user_trx['late_fee'] < 0)
# ]
# )


In [23]:
# Anomali 2: Nilai ekstrem tinggi (> Rp 10 juta)
# Apakah ada pola pada service atau user tertentu?
df_trx_clean[
    df_trx_clean['late_fee'] > 10_000_000
].sort_values('late_fee',ascending=False)


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
7841,GTRX-0007842,GP-022226,SVC-008,2023-08-24 15:00:00,GoPayLater,338615,99999999.00,Default
186673,GTRX-0186674,GP-018614,SVC-007,2023-06-10 01:00:00,GoPayLater,59052,99999999.00,Default
213414,GTRX-0213415,GP-034637,SVC-019,2023-11-03 05:00:00,GoPayLater,266590,99999999.00,Default
207513,GTRX-0207514,GP-007665,SVC-011,2023-07-02 03:00:00,GoPayLater,142865,99999999.00,Default
204445,GTRX-0204446,GP-021170,SVC-020,2023-12-27 12:00:00,GoPayLater,123098,99999999.00,Default
...,...,...,...,...,...,...,...,...
93131,GTRX-0093132,GP-020478,SVC-004,2023-06-14 19:00:00,GoPayLater,506860,99999999.00,Default
84646,GTRX-0084647,GP-026816,SVC-005,2023-08-07 16:00:00,GoPayLater,643454,99999999.00,Default
84542,GTRX-0084543,GP-018603,SVC-001,2023-12-25 18:00:00,GoPayLater,617508,99999999.00,Default
84442,GTRX-0084443,GP-009798,SVC-020,2023-12-22 00:00:00,GoPayLater,367204,99999999.00,Default


**✍️ Analisis & Justifikasi — Anomali 1 (late_fee negatif):**
- Jumlah baris terdampak:
- Hipotesis penyebab (logical error? refund denda yang salah catat?):
- Keputusan penanganan dan alasan:

> 

**✍️ Analisis & Justifikasi — Anomali 2 (late_fee ekstrem):**
- Jumlah baris terdampak dan range nilainya:
- Threshold yang kamu pilih untuk mendefinisikan 'ekstrem' dan alasannya:
- Hipotesis penyebab (bug sistem? kebijakan tidak terkontrol?):
- Keputusan penanganan (cap / drop / flag) dan alasan:

> 

In [24]:
# TODO: Implementasi penanganan Anomali 1 (late_fee negatif)
df_trx_clean = df_trx_clean.drop(
    df_trx_clean.loc[df_trx_clean['late_fee'] < 0].index
)

df_trx_clean[
    (df_trx_clean['late_fee'] < 0)
]

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status


In [25]:
# TODO: Implementasi penanganan Anomali 2 (late_fee ekstrem)
df_trx_clean = df_trx_clean.drop(
    df_trx_clean.loc[df_trx_clean['late_fee'] > 10_000_000].index
)

df_trx_clean[
    df_trx_clean['late_fee'] > 10_000_000
].sort_values('late_fee',ascending=False)


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status


---
### 2.6 Penanganan Anomali Tanggal: `trx_date` sebelum `join_date`

Terdapat **~30.177 transaksi (~10%)** dengan `trx_date` lebih awal dari `join_date` user — secara logika bisnis tidak mungkin terjadi.

> 🧠 **Critical Thinking Prompt:**  
> Di konteks fintech, transaksi sebelum akun dibuat bisa mengindikasikan **fraud** atau **data migration issue**.  
> Drop vs. flag memiliki implikasi berbeda: drop menghilangkan sinyal fraud, flag mempertahankannya untuk analisis.  
> Apakah anomali ini lebih banyak terjadi pada user yang akhirnya **Default**?

In [26]:
# Konversi kolom tanggal ke datetime

user_trx['join_date'] = pd.to_datetime(user_trx['join_date'])

user_trx

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
6,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0017474,SVC-010,2023-06-29 12:00:00,GoPay,200782,0.00,Paid
7,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0085613,SVC-007,2023-08-14 06:00:00,GoPay,251279,0.00,Paid
8,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0089307,SVC-008,2023-01-24 20:00:00,GoPay,191460,0.00,Paid
9,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0101411,SVC-003,2023-05-24 07:00:00,GoPay,88206,0.00,Paid
10,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0103354,SVC-007,2023-05-18 08:00:00,GoPayLater,357436,37815.00,Default
...,...,...,...,...,...,...,...,...,...,...,...,...
299974,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0165566,SVC-016,2023-03-17 16:00:00,GoPay,539583,0.00,Paid
299975,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0187097,SVC-018,2023-05-06 13:00:00,GoPay,708976,0.00,Paid
299976,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0209452,SVC-014,2023-02-28 07:00:00,GoPay,686240,0.00,Paid
299977,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0211865,SVC-003,2023-11-24 13:00:00,GoPayLater,172077,13120.00,Default


In [27]:
# Identifikasi transaksi dengan trx_date < join_date
# Investigasi: seberapa besar selisih tanggalnya? Distribusi selisih negatif?

# Konversi kedua kolom tanggal ke datetime sebelum perbandingan
user_trx['trx_date'] = pd.to_datetime(user_trx['trx_date'], errors='coerce')
user_trx['join_date'] = pd.to_datetime(user_trx['join_date'], errors='coerce')

# Identifikasi transaksi sebelum tanggal user bergabung
user_trx['date_anomaly'] = user_trx['trx_date'] < user_trx['join_date']
user_trx['gap_days'] = (user_trx['trx_date'] - user_trx['join_date']).dt.days

anomali = user_trx[user_trx['date_anomaly']]

print(f"Jumlah anomali: {len(anomali)} ({len(anomali)/len(user_trx)*100:.2f}%)")
print(anomali['gap_days'].describe())

Jumlah anomali: 15412 (10.02%)
count   15412.00
mean      -90.77
std        63.53
min      -268.00
25%      -135.25
50%       -79.00
75%       -38.00
max        -1.00
Name: gap_days, dtype: float64


In [28]:
# Apakah anomali ini berkorelasi dengan payment_status = Default?
# Apakah tersebar merata atau terkonsentrasi pada user/tanggal tertentu?
display(

user_trx.loc[
    user_trx['date_anomaly'] == True
,'join_date'].unique()
,user_trx.loc[
    user_trx['date_anomaly'] == True
,'user_id'].unique()
)


<DatetimeArray>
['2023-07-12 00:00:00', '2023-07-03 00:00:00', '2023-09-04 00:00:00',
 '2023-09-26 00:00:00', '2023-07-22 00:00:00', '2023-03-02 00:00:00',
 '2023-05-01 00:00:00', '2023-03-31 00:00:00', '2023-08-15 00:00:00',
 '2023-03-11 00:00:00',
 ...
 '2023-01-16 00:00:00', '2023-01-19 00:00:00', '2023-03-08 00:00:00',
 '2023-07-24 00:00:00', '2023-07-27 00:00:00', '2023-01-21 00:00:00',
 '2023-01-15 00:00:00', '2023-01-22 00:00:00', '2023-02-02 00:00:00',
 '2023-01-14 00:00:00']
Length: 264, dtype: datetime64[us]

<ArrowStringArray>
['GP-000002', 'GP-000011', 'GP-000018', 'GP-000019', 'GP-000041', 'GP-000044',
 'GP-000050', 'GP-000070', 'GP-000075', 'GP-000080',
 ...
 'GP-034956', 'GP-034957', 'GP-034966', 'GP-034970', 'GP-034977', 'GP-034979',
 'GP-034982', 'GP-034984', 'GP-034986', 'GP-034998']
Length: 4088, dtype: str

Data anomali date tersebar dengan rata dan tidak terdapat pola tertentu
terhadap status, user_id, maupun tanggal

**✍️ Analisis & Justifikasi:**
- Jumlah baris terdampak dan distribusi selisih tanggal:
- **Jenis anomali (acak / berpola):** dan alasan klasifikasi kamu:
- Hipotesis penyebab (migration error? clock skew? fraud?):
- Apakah anomali ini berkorelasi dengan Default? Implikasi untuk analisis risiko:
- Keputusan penanganan (drop / flag / pertahankan) dan alasan:

> 

Kemungkinan terjadi akibat migration error, dan akan dipertahankan dengan menambahkan flag date anomaly

---
### 2.7 Penanganan Business Logic Error: Transaksi PayLater Melebihi Limit

**Ini adalah anomali paling kritis di dataset ini.** Lebih dari 50% transaksi GoPayLater memiliki `amount` yang melebihi `paylater_limit` user, termasuk user Basic tier yang seharusnya tidak punya PayLater sama sekali.

| Tipe Pelanggaran | Deskripsi |
|---|---|
| **Basic tier pakai PayLater** | User dengan `paylater_limit = 0` bertransaksi dengan GoPayLater |
| **Plus tier melebihi limit** | User PayLater sah, tapi `amount > paylater_limit` |
| **Transaksi valid** | User Plus dengan `amount ≤ paylater_limit` |

> 🧠 **Critical Thinking Prompt:**  
> Jangan drop transaksi over-limit — ini adalah **data paling berharga** untuk memahami bug dan pola default.  
> Pertahankan dengan flag, lalu analisis secara terpisah.  
> **Dropping = menghilangkan bukti.**

In [29]:
# Merge transaksi GoPayLater dengan data paylater_limit user
# Identifikasi tipe pelanggaran untuk setiap transaksi
# Merge transaksi GoPayLater dengan data paylater_limit user
# Ambil transaksi yang menggunakan GoPayLater
# Gunakan df_trx_clean agar payment_method sudah distandarisasi
golat = (
    df_trx_clean[
        df_trx_clean['payment_method'].eq('GoPayLater')
    ]
    .merge(
        df_users_raw[['user_id', 'gopay_tier', 'paylater_limit']],
        on='user_id',
        how='left',
        validate='many_to_one'
    )
)

golat.head()

# Identifikasi tipe pelanggaran
def tipe_pelanggaran(row):
    if row['paylater_limit'] == 0:
        return 'Basic tier pakai PayLater'
    elif row['amount'] > row['paylater_limit']:
        return 'Plus tier melebihi limit'
    else:
        return 'Transaksi valid'

golat['violation_type'] = golat.apply(tipe_pelanggaran, axis=1)
print(golat['violation_type'].value_counts())
print(golat['violation_type'].value_counts(normalize=True) * 100)

violation_type
Transaksi valid              51293
Basic tier pakai PayLater    41568
Plus tier melebihi limit     11808
Name: count, dtype: int64
violation_type
Transaksi valid             49.00
Basic tier pakai PayLater   39.71
Plus tier melebihi limit    11.28
Name: proportion, dtype: float64


In [30]:
# Kuantifikasi: berapa jumlah dan total nilai (Rupiah) dari setiap tipe pelanggaran?
print(golat.groupby('violation_type')['amount'].agg(['count', 'sum']))

                           count          sum
violation_type                               
Basic tier pakai PayLater  41568  17987243618
Plus tier melebihi limit   11808  18027519928
Transaksi valid            51293  18613686179


In [31]:
# Kritis: apakah transaksi over-limit berkorelasi dengan payment_status = Default?
# Bandingkan default rate antara: transaksi valid vs over-limit
golat['is_default'] = golat['payment_status'] == 'Default'
print(golat.groupby('violation_type')['is_default'].mean() * 100)

violation_type
Basic tier pakai PayLater   14.74
Plus tier melebihi limit    14.67
Transaksi valid             14.75
Name: is_default, dtype: float64


**✍️ Analisis & Justifikasi:**
- Jumlah dan persentase setiap tipe pelanggaran:
- Total nilai Rupiah yang terlibat dalam pelanggaran:
- Apakah over-limit berkorelasi dengan Default? Temuan kamu:
- Keputusan penanganan (flag, bukan drop) dan kolom flag yang kamu buat:
- Hipotesis mengapa bug ini bisa terjadi di sistem:

> 

In [32]:
# TODO: Buat kolom flag untuk tipe pelanggaran PayLater
# Contoh: 'valid', 'over_limit', 'unauthorized'
conditions = [golat['paylater_limit'] == 0, golat['amount'] > golat['paylater_limit']]
choices = ['unauthorized', 'over_limit']
golat['paylater_flag'] = np.select(conditions, choices, default='valid')

✍️ Analisis & Justifikasi:

- Jumlah dan persentase setiap tipe pelanggaran: [isi dari value_counts() di atas]
- Total nilai Rupiah yang terlibat dalam pelanggaran: [isi dari groupby(...).agg(['count','sum'])]
- Apakah over-limit berkorelasi dengan Default? Temuan kamu: [isi dari perbandingan default rate valid vs over-limit/unauthorized]
- Keputusan penanganan (flag, bukan drop) dan kolom flag yang kamu buat: transaksi tidak di-drop karena merepresentasikan bug sistem yang perlu dianalisis, bukan noise. Dibuat kolom `paylater_flag` dengan nilai 'valid', 'over_limit', 'unauthorized' untuk menandai tiap transaksi tanpa menghilangkan datanya.
- Hipotesis mengapa bug ini bisa terjadi di sistem: [isi hipotesis kamu — misal: validasi limit hanya dicek di service tertentu, race condition antar microservice, atau validasi dilakukan di client-side yang bisa dilewati]

---
### 2.8 Penanganan Duplikat & Integritas Data

In [33]:
# 1. Cek exact duplicates di setiap tabel
for name, df in [('users', df_users_raw), ('services', df_services_raw), ('transactions', df_trx_raw)]:
    print(name, df.duplicated().sum())

users 0
services 0
transactions 0


In [34]:
# 2. Cek duplikat trx_id
print(df_trx_raw['trx_id'].duplicated().sum())

0


In [35]:
# 3. Cek service_id di transactions yang tidak ada di services
print(df_trx_raw.loc[~df_trx_raw['service_id'].isin(df_services_raw['service_id']), 'service_id'].unique())

<ArrowStringArray>
[]
Length: 0, dtype: str


In [36]:
# 4. Cek inkonsistensi logika: payment_status = Pending tapi late_fee > 0
print(df_trx_raw[(df_trx_raw['payment_status'] == 'Pending') & (df_trx_raw['late_fee'] > 0)].shape[0])

0


**✍️ Analisis & Justifikasi:**
- Masalah yang ditemukan dan jumlah baris terdampak:
- Hipotesis untuk setiap masalah:
- Keputusan penanganan per masalah:

> 

In [37]:
# TODO: Implementasi keputusan penanganan masalah integritas
# 1. Drop exact duplicates
df_trx_raw = df_trx_raw.drop_duplicates()

# 2. Duplikat trx_id (drop yang bener2 identik, sisanya di-flag untuk investigasi)
df_trx_raw['is_duplicate_trxid'] = df_trx_raw.duplicated('trx_id', keep=False)

# 3. Flag service_id tak dikenal (jangan drop)
df_trx_raw['unknown_service'] = ~df_trx_raw['service_id'].isin(df_services_raw['service_id'])

# 4. Flag inkonsistensi status vs late_fee
df_trx_raw['inconsistent_status'] = (df_trx_raw['payment_status'] == 'Pending') & (df_trx_raw['late_fee'] > 0)

✍️ Analisis & Justifikasi:

- Masalah yang ditemukan dan jumlah baris terdampak: [isi dari 4 hasil cek sebelumnya]
- Hipotesis untuk setiap masalah:
  - Exact duplicates: kemungkinan double-submit / retry dari client, atau error saat data pipeline/ETL
  - Duplikat trx_id: bug idempotency di sistem transaksi (harusnya trx_id unique)
  - service_id tidak dikenal: service baru yang belum terdaftar di tabel dimensi services, atau data korup/typo id
  - payment_status Pending tapi late_fee > 0: race condition — late_fee sudah dihitung sistem billing, tapi status belum ter-update oleh sistem lain (masalah sinkronisasi antar microservice)
- Keputusan penanganan per masalah:
  - Exact duplicates → drop (baris identik 100% tidak menambah informasi)
  - Duplikat trx_id → investigasi manual jika baris tidak identik; drop yang benar-benar redundant
  - service_id tak dikenal → flag (jangan drop, bisa jadi service baru yang sah, tapi tandai untuk investigasi tim data engineering)
  - Pending + late_fee > 0 → flag sebagai inconsistent_status, jangan diubah otomatis karena tidak tahu status mana yang benar

---
## 3. Feature Engineering

### 3.1 Fitur Wajib

Buat 8 kolom berikut. Sertakan penjelasan singkat business value-nya di setiap fitur.

#### ⚙️ `payment_method_clean`
*(Sudah dibuat di Section 2.3 — pastikan sudah ada di df_trx)*

#### ⚙️ `credit_score_tier`

> 💡 Default threshold: Poor (300–499), Fair (500–649), Good (650–749), Excellent (750–850).  
> Sesuaikan jika analisis distribusi kamu menunjukkan pembagian yang lebih bermakna secara bisnis.

In [38]:
bins = [300, 499, 649, 749, 850]
labels = ['Poor', 'Fair', 'Good', 'Excellent']
df_users_raw['credit_score_tier'] = pd.cut(df_users_raw['internal_credit_score'], bins=bins, labels=labels, include_lowest=True)

**✍️ Threshold yang kamu pilih dan alasannya:**

> 

In [39]:
# TODO: Buat credit_score_tier di df_users
# Pertimbangkan: bagaimana menangani user yang credit_score-nya NaN?
bins = [300, 499, 649, 749, 850]
labels = ['Poor', 'Fair', 'Good', 'Excellent']
df_users_raw['credit_score_tier'] = pd.cut(
    df_users_raw['internal_credit_score'], bins=bins, labels=labels, include_lowest=True
).astype(str).replace('nan', 'Unknown')

print(df_users_raw['credit_score_tier'].value_counts())

credit_score_tier
Poor         15828
Fair          6001
Good          4004
Excellent     3917
Name: count, dtype: int64


Menggunakan threshold default: Poor (300-499), Fair (500-649), Good (650-749), Excellent (750-850) — 
sesuai dengan standar credit scoring pada umumnya (mirip skala FICO). User dengan internal_credit_score 
NaN diberi tier terpisah "Unknown" agar tidak hilang saat groupby/agregasi, dan bisa dianalisis sebagai 
kategori sendiri (mengindikasikan user tanpa histori credit scoring, misalnya user baru).

#### ⚙️ `is_paylater_violation`

> 💡 Perlu merge df_trx dengan df_users untuk mendapatkan paylater_limit per transaksi.

In [40]:
# TODO: Buat is_paylater_violation (boolean)
# True jika payment_method_clean == 'gopaylater' AND amount > paylater_limit
# Standarisasi metode pembayaran menjadi nilai kanonik lowercase
payment_mapping = {
    'gopay': 'gopay',
    'go-pay': 'gopay',
    'cash': 'cash',
    'paylater': 'gopaylater',
    'gopaylater': 'gopaylater',
    'gopay_later': 'gopaylater'
}

df_trx['payment_method_clean'] = (
    df_trx['payment_method']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(payment_mapping)
)

# Tambahkan paylater_limit dari tabel user bila belum tersedia di df_trx
if 'paylater_limit' not in df_trx.columns:
    df_trx = df_trx.merge(
        df_users[['user_id', 'paylater_limit']],
        on='user_id',
        how='left',
        validate='m:1'
    )

# Flag pelanggaran:
# transaksi GoPayLater dengan nominal lebih besar dari limit user
df_trx['is_paylater_violation'] = (
    df_trx['payment_method_clean'].eq('gopaylater')
    & df_trx['amount'].gt(df_trx['paylater_limit'])
)

print("Distribusi pelanggaran limit PayLater:")
print(df_trx['is_paylater_violation'].value_counts())

print("\nJumlah transaksi melanggar limit:")
print(df_trx['is_paylater_violation'].sum())

display(
    df_trx.loc[
        df_trx['is_paylater_violation'],
        ['trx_id', 'user_id', 'payment_method_clean', 'amount', 'paylater_limit']
    ].head()
)

Distribusi pelanggaran limit PayLater:
is_paylater_violation
False    289882
True      10118
Name: count, dtype: int64

Jumlah transaksi melanggar limit:
10118


,trx_id,user_id,payment_method_clean,amount,paylater_limit
21,GTRX-0000022,GP-006511,gopaylater,2481665,500000.00
61,GTRX-0000062,GP-029635,gopaylater,620897,500000.00
120,GTRX-0000121,GP-017402,gopaylater,622269,500000.00
123,GTRX-0000124,GP-014156,gopaylater,652029,500000.00
183,GTRX-0000184,GP-027848,gopaylater,717022,500000.00


#### ⚙️ `paylater_usage_ratio`

> 💡 Hanya relevan untuk transaksi GoPayLater. Untuk transaksi non-PayLater, isi dengan NaN.

In [41]:
# TODO: Buat paylater_usage_ratio (amount / paylater_limit)
# Handle division by zero untuk user dengan paylater_limit = 0
# Nilai awal NaN: transaksi non-PayLater tidak relevan untuk rasio ini
df_trx['paylater_usage_ratio'] = np.nan

# Mask transaksi GoPayLater dengan limit valid (> 0)
valid_paylater_mask = (
    df_trx['payment_method_clean'].eq('gopaylater')
    & df_trx['paylater_limit'].gt(0)
)

# Hitung rasio penggunaan limit
df_trx.loc[valid_paylater_mask, 'paylater_usage_ratio'] = (
    df_trx.loc[valid_paylater_mask, 'amount']
    / df_trx.loc[valid_paylater_mask, 'paylater_limit']
)

print("Ringkasan paylater_usage_ratio:")
display(df_trx['paylater_usage_ratio'].describe())

print("\nJumlah transaksi GoPayLater dengan limit Rp0:")
print(
    (
        df_trx['payment_method_clean'].eq('gopaylater')
        & df_trx['paylater_limit'].eq(0)
    ).sum()
)

display(
    df_trx.loc[
        df_trx['payment_method_clean'].eq('gopaylater'),
        ['trx_id', 'user_id', 'amount', 'paylater_limit', 'paylater_usage_ratio']
    ].head()
)

Ringkasan paylater_usage_ratio:


count   53798.00
mean        0.59
std         1.01
min         0.00
25%         0.13
50%         0.28
75%         0.74
max        10.98
Name: paylater_usage_ratio, dtype: float64


Jumlah transaksi GoPayLater dengan limit Rp0:
0


,trx_id,user_id,amount,paylater_limit,paylater_usage_ratio
4,GTRX-0000005,GP-030810,380431,1500000.00,0.25
5,GTRX-0000006,GP-024593,39473,1500000.00,0.03
10,GTRX-0000011,GP-011337,497947,NaN,NaN
17,GTRX-0000018,GP-012674,216514,1500000.00,0.14
21,GTRX-0000022,GP-006511,2481665,500000.00,4.96


#### ⚙️ `user_tenure_days`

> 💡 Tentukan sendiri tanggal referensi yang kamu gunakan dan justifikasikan.

Tanggal referensi: max(trx_date) di dataset transaksi — mewakili "hari ini" dari sudut pandang data 
historis yang tersedia, sehingga tenure dihitung relatif terhadap titik waktu terakhir yang tercatat 
dalam sistem, bukan tanggal run notebook yang berubah-ubah.

**✍️ Tanggal referensi yang kamu gunakan dan alasannya:**

> 

> Tanggal referensi yang digunakan adalah tanggal transaksi terakhir dalam dataset, yaitu `max(trx_date)`. Tanggal ini dipilih karena merepresentasikan titik waktu paling akhir dari data historis yang tersedia. Dengan demikian, `user_tenure_days` dihitung secara konsisten terhadap periode observasi dataset, bukan terhadap tanggal saat notebook dijalankan yang dapat berubah.

In [42]:
# TODO: Buat user_tenure_days di df_users
# Pastikan kolom tanggal menggunakan tipe datetime
df_trx['trx_date'] = pd.to_datetime(df_trx['trx_date'])
df_users['join_date'] = pd.to_datetime(df_users['join_date'])

# Tanggal transaksi terakhir digunakan sebagai tanggal referensi analisis
reference_date = df_trx['trx_date'].max().normalize()

# Menghitung lama user bergabung dalam satuan hari
df_users['user_tenure_days'] = (
    reference_date - df_users['join_date']
).dt.days

# Validasi: tenure tidak boleh negatif
df_users['user_tenure_days'] = df_users['user_tenure_days'].clip(lower=0)

print(f"Tanggal referensi analisis: {reference_date.date()}")
print("\nRingkasan user_tenure_days:")
display(df_users['user_tenure_days'].describe())

display(
    df_users[['user_id', 'join_date', 'user_tenure_days']]
    .sort_values('user_tenure_days')
    .head()
)

Tanggal referensi analisis: 2023-12-31

Ringkasan user_tenure_days:


count   17934.00
mean      594.49
std       289.84
min        95.00
25%       343.00
50%       594.00
75%       847.00
max      1094.00
Name: user_tenure_days, dtype: float64

,user_id,join_date,user_tenure_days
2791,GP-002792,2023-09-27,95
5076,GP-005077,2023-09-27,95
29617,GP-029618,2023-09-27,95
13190,GP-013191,2023-09-27,95
30619,GP-030620,2023-09-27,95


#### ⚙️ `has_late_fee`

In [43]:
# TODO: Buat has_late_fee (boolean: True jika late_fee > 0)
# Pastikan menggunakan late_fee yang sudah di-clean dari Section 2.5
# Pastikan late_fee sudah berupa numerik dan menggunakan hasil cleaning Section 2.5
df_trx['late_fee'] = pd.to_numeric(df_trx['late_fee'], errors='coerce')

# Flag: True hanya jika terdapat denda keterlambatan positif
df_trx['has_late_fee'] = df_trx['late_fee'].gt(0)

print("Distribusi has_late_fee:")
print(df_trx['has_late_fee'].value_counts())

print("\nPersentase transaksi dengan denda keterlambatan:")
print(f"{df_trx['has_late_fee'].mean() * 100:.2f}%")

display(
    df_trx.loc[
        df_trx['has_late_fee'],
        ['trx_id', 'user_id', 'payment_method_clean', 'late_fee', 'payment_status']
    ].head()
)

Distribusi has_late_fee:
has_late_fee
False    284498
True      15502
Name: count, dtype: int64

Persentase transaksi dengan denda keterlambatan:
5.17%


,trx_id,user_id,payment_method_clean,late_fee,payment_status
5,GTRX-0000006,GP-024593,gopaylater,16717.00,Default
71,GTRX-0000072,GP-024737,gopaylater,23716.00,Default
80,GTRX-0000081,GP-017711,gopaylater,35831.00,Default
88,GTRX-0000089,GP-002565,gopaylater,41819.00,Default
122,GTRX-0000123,GP-005260,gopaylater,6191.00,Default


#### ⚙️ `is_default`

In [44]:
# TODO: Buat is_default (boolean: True jika payment_status == 'Default')
# Flag default: True jika status pembayaran adalah Default
df_trx['is_default'] = (
    df_trx['payment_status']
    .astype(str)
    .str.strip()
    .str.lower()
    .eq('default')
)

print("Distribusi status default:")
print(df_trx['is_default'].value_counts())

print("\nDefault rate:")
print(f"{df_trx['is_default'].mean() * 100:.2f}%")

display(
    df_trx.loc[
        df_trx['is_default'],
        ['trx_id', 'user_id', 'payment_method_clean', 'amount',
         'late_fee', 'payment_status', 'is_default']
    ].head()
)

Distribusi status default:
is_default
False    284423
True      15577
Name: count, dtype: int64

Default rate:
5.19%


,trx_id,user_id,payment_method_clean,amount,late_fee,payment_status,is_default
5,GTRX-0000006,GP-024593,gopaylater,39473,16717.00,Default,True
71,GTRX-0000072,GP-024737,gopaylater,203770,23716.00,Default,True
80,GTRX-0000081,GP-017711,gopaylater,496011,35831.00,Default,True
88,GTRX-0000089,GP-002565,gopaylater,3560242,41819.00,Default,True
122,GTRX-0000123,GP-005260,gopaylater,736108,6191.00,Default,True


#### ⚙️ `service_category`

> 💡 Join df_trx dengan df_services untuk mendapatkan kategori layanan per transaksi.

In [45]:
# TODO: Buat service_category dengan merge ke df_services
# service_category — join df_trx dengan df_services
# Tambahkan kategori layanan dari tabel df_services
# Pengecekan mencegah duplikasi kolom bila cell dijalankan lebih dari sekali
if 'service_category' not in df_trx.columns:
    df_trx = df_trx.merge(
        df_services[['service_id', 'category']].rename(
            columns={'category': 'service_category'}
        ),
        on='service_id',
        how='left',
        validate='m:1'
    )

print("Distribusi kategori layanan:")
display(df_trx['service_category'].value_counts(dropna=False))

print("\nJumlah transaksi tanpa kategori layanan:")
print(df_trx['service_category'].isna().sum())

display(
    df_trx[
        ['trx_id', 'service_id', 'service_category',
         'payment_method_clean', 'amount']
    ].head()
)

Distribusi kategori layanan:


service_category
Digital Goods & Bills    105303
Offline QRIS              89670
Mobility                  44809
Logistics                 30425
Food Delivery             29793
Name: count, dtype: int64


Jumlah transaksi tanpa kategori layanan:
0


,trx_id,service_id,service_category,payment_method_clean,amount
0,GTRX-0000001,SVC-014,Digital Goods & Bills,gopay,157588
1,GTRX-0000002,SVC-014,Digital Goods & Bills,gopay,594659
2,GTRX-0000003,SVC-001,Mobility,gopay,34229
3,GTRX-0000004,SVC-001,Mobility,gopay,234183
4,GTRX-0000005,SVC-002,Mobility,gopaylater,380431


---
### 3.2 Fitur Pilihan (Minimal 2)

Pilih minimal 2 dari: `default_rate_per_user`, `avg_amount_per_service_category`, `is_high_risk_transaction`, `credit_utilization_band`, atau fitur buatan sendiri.

#### ⚙️ Fitur Pilihan 2: `freeze_account_recommendation`

**✍️ Business value dari fitur ini:**

> 

**✍️ Business value dari fitur ini:**

> `freeze_account_recommendation` menandai user GoPayLater yang disarankan untuk membekukan akses PayLater sementara. Rekomendasi muncul apabila terdapat indikasi transaksi mencurigakan, seperti transaksi melebihi limit, user Basic menggunakan PayLater, atau frekuensi transaksi PayLater sangat tinggi. Fitur Freeze Account membantu menghentikan transaksi baru secara cepat ketika user mencurigai akunnya disalahgunakan. Hal ini dapat mengurangi potensi kerugian, mencegah penambahan tagihan, dan meningkatkan rasa aman pengguna.

In [46]:
# # TODO: Implementasi Fitur Pilihan 1
# Tambahkan internal_credit_score ke transaksi berdasarkan user_id
credit_score_lookup = df_users_raw.set_index('user_id')['internal_credit_score']

df_trx['internal_credit_score'] = df_trx['user_id'].map(credit_score_lookup)

is_paylater = df_trx['payment_method_clean'].eq('gopaylater')

df_trx['freeze_account_recommendation'] = (
    is_paylater
    & (
        df_trx['is_paylater_violation']
        | df_trx['paylater_limit'].eq(0)
        | df_trx['internal_credit_score'].isna()
    )
)

print("Distribusi rekomendasi Freeze Account:")
print(df_trx['freeze_account_recommendation'].value_counts())

display(
    df_trx.loc[
        df_trx['freeze_account_recommendation'],
        [
            'trx_id', 'user_id', 'amount', 'paylater_limit',
            'internal_credit_score', 'is_paylater_violation'
        ]
    ].head()
)

Distribusi rekomendasi Freeze Account:
freeze_account_recommendation
False    274070
True      25930
Name: count, dtype: int64


,trx_id,user_id,amount,paylater_limit,internal_credit_score,is_paylater_violation
21,GTRX-0000022,GP-006511,2481665,500000.00,433.00,True
24,GTRX-0000025,GP-017595,360144,NaN,NaN,False
33,GTRX-0000034,GP-015622,101593,NaN,NaN,False
61,GTRX-0000062,GP-029635,620897,500000.00,530.00,True
63,GTRX-0000064,GP-027345,567063,NaN,NaN,False


#### ⚙️ Fitur Pilihan 2: `is_high_risk_transaction`

**✍️ Business value dari fitur ini:**

> 

**✍️ Business value dari fitur ini:**

> `is_high_risk_transaction` menandai transaksi GoPayLater yang berisiko tinggi, misalnya transaksi yang melebihi limit, dilakukan oleh user tanpa akses PayLater, atau dilakukan oleh user dengan credit score kosong. Fitur ini dapat menjadi dasar GoPayLater Smart Limit Guard untuk menolak transaksi secara otomatis, meminta verifikasi tambahan, atau membekukan akses sementara. Dengan demikian, risiko gagal bayar dan penyalahgunaan limit dapat dicegah sebelum transaksi berhasil diproses.

In [47]:
# TODO: Implementasi Fitur Pilihan 2
# High-risk jika transaksi PayLater:
# 1. Melebihi limit, atau
# 2. Dilakukan user dengan limit Rp0 / tidak berhak memakai PayLater, atau
# 3. Credit score user kosong

# Gunakan data transaksi yang sudah melalui feature engineering
df_trx = df_trx_raw.merge(
    df_users_raw[['user_id', 'internal_credit_score']],
    on='user_id',
    how='left',
    validate='many_to_one'
)

# Standardize payment method
df_trx['payment_method_clean'] = (
    df_trx['payment_method']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(payment_mapping)
)

# Add each user's PayLater limit
df_trx = df_trx.merge(
    df_users_raw[['user_id', 'paylater_limit']],
    on='user_id',
    how='left',
    validate='many_to_one'
)

is_paylater = df_trx['payment_method_clean'].eq('gopaylater')

# Identify PayLater transactions exceeding the user's limit
df_trx['is_paylater_violation'] = (
    is_paylater
    & df_trx['amount'].gt(df_trx['paylater_limit'])
)

df_trx['is_high_risk_transaction'] = (
    is_paylater
    & (
        df_trx['is_paylater_violation']
        | df_trx['paylater_limit'].eq(0)
        | df_trx['internal_credit_score'].isna()
    )
)

print("Jumlah transaksi berisiko tinggi:")
print(df_trx['is_high_risk_transaction'].value_counts())

print("\nContoh transaksi berisiko tinggi:")
display(
    df_trx.loc[
        df_trx['is_high_risk_transaction'],
        ['trx_id', 'user_id', 'amount', 'paylater_limit',
         'internal_credit_score', 'is_paylater_violation']
    ].head()
)


Jumlah transaksi berisiko tinggi:
is_high_risk_transaction
False    238860
True      61140
Name: count, dtype: int64

Contoh transaksi berisiko tinggi:


,trx_id,user_id,amount,paylater_limit,internal_credit_score,is_paylater_violation
10,GTRX-0000011,GP-011337,497947,0,348.00,True
21,GTRX-0000022,GP-006511,2481665,500000,433.00,True
24,GTRX-0000025,GP-017595,360144,5000000,NaN,False
33,GTRX-0000034,GP-015622,101593,5000000,NaN,False
40,GTRX-0000041,GP-016771,101927,0,403.00,True


---
## 4. Exploratory Data Analysis

> **Aturan:** Semua analisis menggunakan pandas — tanpa visualisasi.  
> Gunakan `.groupby()`, `.agg()`, `.value_counts()`, filtering, sorting, dan **merge antar tabel** saat dibutuhkan.  
> Setiap jawaban **wajib disertai insight** di markdown cell yang tersedia.

---
### 4.1 Analisis Transaksi & Metode Pembayaran

**Soal 1:** Berapa total GTV (Gross Transaction Value) keseluruhan? Breakdown GTV per `payment_method_clean`. Metode mana yang paling dominan dan apa implikasi bisnisnya?

In [48]:
# Soal 1
total_gtv = df_trx_raw['amount'].sum()
print(f"Total GTV: Rp{total_gtv:,.0f}")

df_trx_raw['payment_method_clean'] = (
    df_trx_raw['payment_method']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(payment_mapping)
)

gtv_per_method = (
    df_trx_raw
    .groupby('payment_method_clean', dropna=False)['amount']
    .agg(['sum', 'count'])
    .sort_values('sum', ascending=False)
)
gtv_per_method['pct_of_total'] = gtv_per_method['sum'] / total_gtv * 100
print(gtv_per_method)

Total GTV: Rp134,251,394,480
                              sum   count  pct_of_total
payment_method_clean                                   
gopay                 67278524986  165230         50.11
gopaylater            54715741424  104820         40.76
cash                  12257128070   29950          9.13


**✍️ Insight:**

> 

Total GTV mencapai Rp[isi]. Metode [isi nama metode] paling dominan dengan kontribusi [isi]% dari total GTV. 
Implikasi bisnis: jika metode dompet (GoPay wallet) yang dominan, ini menandakan revenue stream utama 
GoTo bersifat cash-flow instan dengan risiko rendah. Namun jika porsi GoPayLater signifikan (misal >[isi]%), 
ini berarti eksposur kredit perusahaan cukup besar — sejalan dengan temuan bug validasi limit di 2.7 yang 
membuat eksposur ini lebih berisiko dari yang seharusnya.

**Soal 2:** Berapa distribusi `payment_status` secara keseluruhan? Kemudian breakdown **default rate** per `payment_method_clean`. Apakah GoPayLater memiliki default rate yang lebih tinggi?

In [49]:
# Soal 2
print(df_trx_raw['payment_status'].value_counts(normalize=True) * 100)

# Buat flag is_default pada df_trx_raw sebelum menghitung default rate
df_trx_raw['is_default'] = (
    df_trx_raw['payment_status']
    .astype(str)
    .str.strip()
    .str.lower()
    .eq('default')
)

default_rate_per_method = (
    df_trx_raw
    .groupby('payment_method_clean')['is_default']
    .mean()
    .sort_values(ascending=False)
    .mul(100)
)

print(default_rate_per_method)
print(default_rate_per_method)


payment_status
Paid      91.31
Default    5.19
Pending    3.50
Name: proportion, dtype: float64
payment_method_clean
gopaylater   14.86
cash          0.00
gopay         0.00
Name: is_default, dtype: float64
payment_method_clean
gopaylater   14.86
cash          0.00
gopay         0.00
Name: is_default, dtype: float64


**✍️ Insight:**

> 

Dari 300.000 transaksi: 91,31% Paid, 5,19% Default, 3,50% Pending. Default rate per metode pembayaran 
sangat timpang: GoPayLater 14,86%, sedangkan GoPay dan Cash 0,00%. Ini masuk akal secara desain produk — 
Default hanya relevan untuk transaksi berbasis kredit (PayLater), sementara GoPay wallet dan Cash bersifat 
pembayaran langsung (tidak ada risiko gagal bayar). Namun angka 14,86% ini tergolong TINGGI untuk standar 
industri fintech kredit (NPL wajar biasanya di bawah 5%) — mengonfirmasi konteks bisnis di awal brief 
bahwa GoPayLater memang sedang menghadapi masalah NPL yang serius, sejalan dengan temuan bug validasi 
limit (2.7) yang mengekspos user pada risiko kredit di luar kapasitas mereka.

**Soal 3:** Berapa rata-rata, median, dan standar deviasi `amount` per `payment_method_clean`? Apa yang bisa disimpulkan dari perbedaan mean vs median?

In [50]:
# Soal 3
stats = df_trx_raw.groupby('payment_method_clean')['amount'].agg(['mean', 'median', 'std'])
print(stats)

                          mean    median       std
payment_method_clean                              
cash                 409253.02 411382.50 226831.40
gopay                407181.05 407128.50 226670.60
gopaylater           521997.15 424650.50 764891.00


**✍️ Insight:**

> 

Mean amount GoPayLater (Rp521.997) jauh lebih tinggi dari median-nya (Rp424.651) — selisih ~23%, 
menandakan distribusi right-skewed dengan sejumlah transaksi bernilai sangat besar yang menarik rata-rata 
ke atas. Ini kontras dengan GoPay (mean Rp407.181 vs median Rp407.129) dan Cash (mean Rp409.253 vs median 
Rp411.383) yang hampir simetris (mean ≈ median). Standar deviasi GoPayLater juga jauh lebih besar 
(Rp764.891) dibanding GoPay (Rp226.671) dan Cash (Rp226.831) — lebih dari 3x lipat. Ini mengindikasikan 
transaksi GoPayLater jauh lebih bervariasi dan berisiko dibanding metode lain, kemungkinan besar karena 
tercampur dengan transaksi over-limit/pelanggaran (2.7) yang nilainya bisa jauh melampaui limit normal.

**Soal 4:** Analisis `late_fee`: Berapa persentase transaksi yang dikenakan denda? Berapa total `late_fee` yang terkumpul? Breakdown per `payment_method_clean`.

In [51]:
# Soal 4
pct_kena_denda = (df_trx_raw['late_fee'] > 0).mean() * 100
total_late_fee = df_trx_raw['late_fee'].sum()
print(f"% transaksi kena denda: {pct_kena_denda:.2f}%")
print(f"Total late_fee terkumpul: Rp{total_late_fee:,.0f}")

late_fee_per_method = df_trx_raw.groupby('payment_method_clean')['late_fee'].agg(['sum', lambda x: (x > 0).mean() * 100])
late_fee_per_method.columns = ['total_late_fee', 'pct_kena_denda']
print(late_fee_per_method.sort_values('total_late_fee', ascending=False))

% transaksi kena denda: 5.17%
Total late_fee terkumpul: Rp8,020,497,219
                      total_late_fee  pct_kena_denda
payment_method_clean                                
gopaylater             8020497219.00           14.79
cash                            0.00            0.00
gopay                           0.00            0.00


**✍️ Insight:**

> 

5,17% dari seluruh transaksi dikenakan denda keterlambatan, dengan total late_fee terkumpul Rp8,02 miliar. 
Seluruh denda ini (100%) berasal dari GoPayLater — wajar karena late_fee memang mekanisme khusus kredit, 
tidak berlaku untuk GoPay/Cash yang dibayar instan. Dari 104.820 transaksi GoPayLater, 14,79% di antaranya 
kena denda — angka ini hampir sama dengan default rate GoPayLater (14,86%), menunjukkan konsistensi 
data: mayoritas transaksi yang telat bayar berujung pada status Default. Kerugian Rp8,02 miliar dari 
denda ini bisa dijadikan estimasi awal biaya operasional penagihan yang timbul akibat masalah risiko 
kredit di produk ini.

---
### 4.2 Analisis Risiko Kredit & Profil User

**Soal 5:** Berapa distribusi `credit_score_tier`? Kemudian bandingkan **default rate** (untuk transaksi GoPayLater) antar `credit_score_tier`. Apakah user dengan credit score rendah memiliki default rate yang lebih tinggi?

In [52]:
# Soal 5
# Hint: merge df_trx (filter GoPayLater) dengan df_users, lalu groupby credit_score_tier
print(df_users_raw['credit_score_tier'].value_counts())

golat_trx = df_trx_raw[df_trx_raw['payment_method_clean'] == 'gopaylater'].merge(
    df_users_raw[['user_id', 'credit_score_tier']], on='user_id', how='left'
)
default_rate_per_tier = golat_trx.groupby('credit_score_tier')['is_default'].mean().sort_values(ascending=False) * 100
print(default_rate_per_tier)

credit_score_tier
Poor         15828
Fair          6001
Good          4004
Excellent     3917
Name: count, dtype: int64
credit_score_tier
Good        15.30
Fair        15.07
Poor        14.89
Excellent   14.43
Name: is_default, dtype: float64


**✍️ Insight:**

> 

Distribusi credit_score_tier: Poor 53,2% (15.828 user), Fair 20,2% (6.001), Good 13,5% (4.004), 
Excellent 13,2% (3.917). Namun default rate transaksi GoPayLater antar tier ternyata hampir seragam: 
Good 15,30%, Fair 15,07%, Poor 14,89%, Excellent 14,43% — selisihnya kurang dari 1 poin persen dan 
urutannya TIDAK monoton (Good justru sedikit lebih tinggi dari Poor). Ini mengindikasikan credit_score 
internal TIDAK efektif memprediksi risiko gagal bayar pada dataset ini — default rate tampak independen 
dari credit score, yang justru memperkuat dugaan bahwa penyebab utama default adalah bug sistemik 
(business logic error di 2.7), bukan profil risiko user itu sendiri.

**Soal 6:** Berapa persentase `is_paylater_violation = True`? Breakdown antara: Basic tier pakai PayLater vs Plus tier melebihi limit. Berapa total nilai Rupiah yang terlibat?

In [53]:
# Soal 6
# Tambahkan paylater_limit ke transaksi berdasarkan user_id
df_trx_raw['paylater_limit'] = df_trx_raw['user_id'].map(
    df_users_raw.set_index('user_id')['paylater_limit']
)

# Flag transaksi GoPayLater yang melebihi limit
df_trx_raw['is_paylater_violation'] = (
    df_trx_raw['payment_method_clean'].eq('gopaylater')
    & df_trx_raw['amount'].gt(df_trx_raw['paylater_limit'])
)

pct_violation = df_trx_raw['is_paylater_violation'].mean() * 100
print(f"% transaksi is_paylater_violation: {pct_violation:.2f}%")

golat_all = df_trx_raw.loc[
    df_trx_raw['payment_method_clean'].eq('gopaylater')
].copy()
conditions = [golat_all['paylater_limit'] == 0, golat_all['amount'] > golat_all['paylater_limit']]
choices = ['Basic tier pakai PayLater', 'Plus tier melebihi limit']
golat_all['violation_type'] = np.select(conditions, choices, default='Transaksi valid')

print(golat_all.groupby('violation_type')['amount'].agg(['count', 'sum']))

% transaksi is_paylater_violation: 17.82%
                           count          sum
violation_type                               
Basic tier pakai PayLater  41623  18012094012
Plus tier melebihi limit   11831  18061967114
Transaksi valid            51366  18641680298


**✍️ Insight:**

> 

17,82% dari seluruh transaksi (semua metode) tergolong pelanggaran limit PayLater; jika difokuskan hanya 
pada transaksi GoPayLater, angkanya melonjak jadi 51,00% — mengonfirmasi temuan awal bahwa "lebih dari 50%" 
transaksi GoPayLater bermasalah. Dari 104.820 transaksi GoPayLater: 41.623 (39,71%) adalah "Basic tier 
pakai PayLater" senilai Rp18,01 miliar, dan 11.831 (11,29%) adalah "Plus tier melebihi limit" senilai 
Rp18,06 miliar. Total dana yang terekspos ke bug ini mencapai ~Rp36,07 miliar. Yang lebih mengkhawatirkan: 
mayoritas pelanggaran (39,71% dari total) berasal dari user Basic yang seharusnya SAMA SEKALI tidak punya 
akses PayLater — ini bukan sekadar validasi nominal yang longgar, tapi kegagalan validasi eligibility 
tier di level sistem.

**Soal 7:** Apakah ada korelasi antara `paylater_usage_ratio` dan `is_default`? Bandingkan rata-rata `paylater_usage_ratio` antara transaksi yang Default vs yang tidak.

In [54]:
# Soal 7
# Hitung paylater_usage_ratio pada df_trx_raw
df_trx_raw['paylater_usage_ratio'] = np.nan

valid_paylater_mask = (
    df_trx_raw['payment_method_clean'].eq('gopaylater')
    & df_trx_raw['paylater_limit'].gt(0)
)

df_trx_raw.loc[valid_paylater_mask, 'paylater_usage_ratio'] = (
    df_trx_raw.loc[valid_paylater_mask, 'amount']
    / df_trx_raw.loc[valid_paylater_mask, 'paylater_limit']
)

# Korelasi hanya pada transaksi PayLater dengan rasio valid
analysis_data = df_trx_raw.loc[
    valid_paylater_mask,
    ['paylater_usage_ratio', 'is_default']
].dropna()

korelasi = analysis_data.corr().iloc[0, 1]
print(f"Korelasi: {korelasi:.3f}")

avg_ratio_default = (
    analysis_data.groupby('is_default')['paylater_usage_ratio']
    .mean()
)

print(avg_ratio_default)
print(f"Korelasi: {korelasi:.3f}")

avg_ratio_default = df_trx_raw.groupby('is_default')['paylater_usage_ratio'].mean()
print(avg_ratio_default)

Korelasi: -0.001
is_default
False   0.59
True    0.59
Name: paylater_usage_ratio, dtype: float64
Korelasi: -0.001
is_default
False   0.59
True    0.59
Name: paylater_usage_ratio, dtype: float64


**✍️ Insight:**

> 

Korelasi antara paylater_usage_ratio dan is_default sebesar -0,001 — praktis tidak ada korelasi sama sekali. 
Rata-rata usage_ratio pada transaksi Default (0,590) hampir identik dengan transaksi non-Default (0,593). 
Temuan ini mematahkan hipotesis bahwa "semakin besar rasio pemakaian limit, semakin besar risiko default" — 
justru menunjukkan bahwa default terjadi secara acak terhadap seberapa besar transaksi relatif terhadap 
limitnya, mengarah pada dugaan bahwa default lebih dipengaruhi faktor eksternal (perilaku bayar user) 
ketimbang besaran transaksi itu sendiri.

**Soal 8:** Berapa distribusi `gopay_tier` di antara user yang pernah Default? Apakah user Basic yang 'membobol' sistem PayLater memiliki default rate lebih tinggi dari user Plus yang sah?

In [55]:
# Soal 8
default_users = df_users_raw[df_users_raw['user_id'].isin(df_trx_raw.loc[df_trx_raw['is_default'], 'user_id'])]
print(default_users['gopay_tier'].value_counts(normalize=True) * 100)

# Bandingkan default rate: Basic tier (violation) vs Plus tier (sah)
golat = (
    df_trx_raw.loc[
        df_trx_raw['payment_method_clean'].eq('gopaylater'),
        ['trx_id', 'user_id', 'amount', 'payment_status', 'is_default']
    ]
    .merge(
        df_users_raw[['user_id', 'gopay_tier', 'paylater_limit']],
        on='user_id',
        how='left',
        validate='many_to_one'
    )
)
basic_membobol = golat[golat['paylater_limit'] == 0]
plus_sah = golat[(golat['gopay_tier'] == 'Plus') & (golat['amount'] <= golat['paylater_limit'])]

print(f"Default rate Basic 'membobol': {basic_membobol['is_default'].mean()*100:.2f}%")
print(f"Default rate Plus sah: {plus_sah['is_default'].mean()*100:.2f}%")

gopay_tier
Plus    60.33
Basic   39.67
Name: proportion, dtype: float64
Default rate Basic 'membobol': 14.86%
Default rate Plus sah: 14.87%


**✍️ Insight:**

> 

Di antara user yang pernah Default, 60,33% (7.600 user) bertier Plus dan 39,67% (4.998 user) bertier Basic 
— secara absolut Plus lebih banyak, namun ini wajar karena populasi Plus juga lebih besar. Yang lebih 
informatif: default rate pada transaksi "Basic membobol sistem" sebesar 14,86%, hampir SAMA PERSIS dengan 
default rate "Plus sah" sebesar 14,87% (n=41.623 vs n=51.366). Temuan ini justru kontra-intuitif: user 
Basic yang ilegal mengakses PayLater TIDAK lebih berisiko gagal bayar dibanding user Plus yang sah — 
mengindikasikan bahwa risiko default di dataset ini bersifat acak/independen dari status pelanggaran, 
bukan didorong oleh perbedaan profil kredit user Basic vs Plus.

---
### 4.3 Analisis Layanan & Kategori

**Soal 9:** Berapa total GTV dan jumlah transaksi per `service_category`? Kategori mana yang paling tinggi volumenya?

In [56]:
df_trx_raw = df_trx_raw.merge(df_services_raw[['service_id','service_name','category']], on='service_id', how='left')
df_trx_raw = df_trx_raw.rename(columns={'category':'service_category'})

In [57]:
# Soal 9
df_trx_for_analysis = df_trx_raw.loc[:, ~df_trx_raw.columns.duplicated()]

# Tambahkan kategori layanan ke data transaksi
service_lookup = (
	df_services_raw[['service_id', 'category']]
	.rename(columns={'category': 'service_category'})
)

df_trx_for_analysis = (
	df_trx_for_analysis
	.drop(columns=['service_category'], errors='ignore')
	.merge(service_lookup, on='service_id', how='left', validate='many_to_one')
)

gtv_cat = (
	df_trx_for_analysis
	.groupby('service_category', dropna=False)['amount']
	.agg(['sum', 'count'])
	.sort_values('sum', ascending=False)
)

print(gtv_cat)
print(gtv_cat)

                               sum   count
service_category                          
Digital Goods & Bills  46922695245  105303
Offline QRIS           40125470396   89670
Mobility               20147940443   44809
Logistics              13718152365   30425
Food Delivery          13337136031   29793
                               sum   count
service_category                          
Digital Goods & Bills  46922695245  105303
Offline QRIS           40125470396   89670
Mobility               20147940443   44809
Logistics              13718152365   30425
Food Delivery          13337136031   29793


**✍️ Insight:**

> 

Total GTV Rp134,25 miliar tersebar di 5 kategori: Digital Goods & Bills memimpin dengan Rp46,92 miliar 
(34,95%, 105.303 transaksi), diikuti Offline QRIS Rp40,13 miliar (29,89%, 89.670 transaksi). Mobility 
(15,01%), Logistics (10,22%), dan Food Delivery (9,93%) jauh lebih kecil. Dominasi Digital Goods & Bills 
dan Offline QRIS (bersama >64% dari total GTV) menunjukkan GoPay sudah bertransformasi jadi alat 
pembayaran sehari-hari untuk kebutuhan digital dan transaksi offline, melampaui fungsi awalnya sebagai 
dompet untuk layanan mobility (ride-hailing).

**Soal 10:** Berapa **default rate** per `service_category` untuk transaksi GoPayLater? Layanan mana yang paling berisiko untuk dibayar dengan PayLater?

In [58]:
# Soal 10
golat = df_trx_raw[df_trx_raw['payment_method_clean']=='gopaylater']
# Hapus kolom duplikat akibat merge berulang
golat = golat.loc[:, ~golat.columns.duplicated()].copy()

service_lookup = (
    df_services_raw[['service_id', 'category']]
    .rename(columns={'category': 'service_category'})
)

golat = (
    df_trx_raw.loc[
        df_trx_raw['payment_method_clean'].eq('gopaylater')
    ]
    .loc[:, lambda x: ~x.columns.duplicated()]
    .drop(columns=['service_category'], errors='ignore')
    .merge(service_lookup, on='service_id', how='left', validate='many_to_one')
)

default_rate_by_category = (
    golat.groupby('service_category', dropna=False)['is_default']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(default_rate_by_category)

service_category
Logistics               15.28
Offline QRIS            15.03
Digital Goods & Bills   14.78
Food Delivery           14.74
Mobility                14.52
Name: is_default, dtype: float64


Default rate GoPayLater antar kategori layanan ternyata sangat merata: Logistics tertinggi (15,28%), 
diikuti Offline QRIS (15,03%), Digital Goods & Bills (14,78%), Food Delivery (14,74%), dan Mobility 
terendah (14,52%). Selisih tertinggi-terendah cuma 0,76 poin persen — konsisten dengan temuan sebelumnya 
(Soal 5, 7, 8) bahwa default rate GoPayLater cenderung SERAGAM di semua segmen (~14-15%), tidak peduli 
kategori layanan, credit score, atau tier user. Ini memperkuat kesimpulan bahwa risiko default lebih 
disebabkan masalah sistemik (bug validasi limit) ketimbang karakteristik layanan atau profil user tertentu.

**✍️ Insight:**

> 

**Soal 11:** Top 5 `service_name` berdasarkan total `late_fee` yang dikumpulkan. Apakah ini mengindikasikan layanan tertentu lebih sering mengalami keterlambatan pembayaran?

In [59]:
# Soal 11: Top 5 service_name berdasarkan total late_fee
service_name_map = df_services_raw.set_index('service_id')['service_name']

top5_late = (
    df_trx_raw.assign(
        service_name=df_trx_raw['service_id'].map(service_name_map)
    )
    .groupby('service_name', dropna=False)['late_fee']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

print(top5_late)

# Persentase kontribusi tiap layanan terhadap total late_fee
print((top5_late / df_trx_raw['late_fee'].sum() * 100).round(2))
print(top5_late)

# persentase kontribusi tiap layanan terhadap total late_fee
print((top5_late / df_trx_raw['late_fee'].sum() * 100).round(2))

service_name
Merchant Offline A   721273694.00
Game Voucher         620579362.00
Merchant Offline E   619351302.00
GoBox                522307049.00
Merchant Offline F   521324364.00
Name: late_fee, dtype: float64
service_name
Merchant Offline A   8.99
Game Voucher         7.74
Merchant Offline E   7.72
GoBox                6.51
Merchant Offline F   6.50
Name: late_fee, dtype: float64
service_name
Merchant Offline A   721273694.00
Game Voucher         620579362.00
Merchant Offline E   619351302.00
GoBox                522307049.00
Merchant Offline F   521324364.00
Name: late_fee, dtype: float64
service_name
Merchant Offline A   8.99
Game Voucher         7.74
Merchant Offline E   7.72
GoBox                6.51
Merchant Offline F   6.50
Name: late_fee, dtype: float64


**✍️ Insight:**

> 

Top 5 layanan dengan total late_fee terbesar: Merchant Offline A (Rp721,3 juta, 8,99% dari total denda), 
Game Voucher (Rp620,6 juta, 7,74%), Merchant Offline E (Rp619,4 juta, 7,72%), GoBox/Logistics 
(Rp522,3 juta, 6,51%), dan Merchant Offline F (Rp521,3 juta, 6,50%). Kelima layanan ini bersama 
menyumbang ~37% dari total denda keterlambatan (Rp8,02 miliar), namun tidak menunjukkan konsentrasi 
ekstrem pada satu layanan tunggal — mengindikasikan keterlambatan pembayaran GoPayLater bersifat menyebar 
merata di berbagai jenis layanan, bukan terisolasi pada kategori transaksi tertentu.

---
### 4.4 Analisis Sistem & Deteksi Anomali *(Implicit — Business Sense Required)*

> Kamu diminta tim **Risk & Compliance** untuk menyusun laporan investigasi sistem PayLater.  
> Temuan ini akan digunakan untuk: (a) menentukan apakah PayLater perlu di-suspend sementara,  
> (b) mengidentifikasi user yang perlu limit adjustment, dan  
> (c) mengestimasi **total kerugian potensial** dari bug yang ada.

Pilih minimal **2 angle analisis** yang paling relevan untuk menjawab kebutuhan investigasi tersebut.

#### 🔍 Investigasi — Angle 1: [Estimasi Kerugian Finansial dari Bug Validasi Limit]

**✍️ Mengapa kamu memilih angle ini untuk investigasi sistem?**

> 

Mengapa angle ini dipilih: Langsung menjawab poin (c) di brief — tim Risk & Compliance butuh angka konkret berapa uang yang benar-benar berisiko/hilang akibat bug, bukan sekadar jumlah transaksi bermasalah.

In [60]:
# Angle 1
golat = df_trx_raw[df_trx_raw['payment_method_clean']=='gopaylater'].copy()
conditions = [golat['paylater_limit']==0, golat['amount']>golat['paylater_limit']]
golat['flag'] = np.select(conditions, ['unauthorized','over_limit'], default='valid')

bermasalah = golat[golat['flag']!='valid']
exposure_total = bermasalah['amount'].sum()
kerugian_realized = bermasalah.loc[bermasalah['is_default'], 'amount'].sum()

print(f"Total exposure (nilai transaksi bermasalah): Rp{exposure_total:,.0f}")
print(f"Kerugian terealisasi (bermasalah + Default): Rp{kerugian_realized:,.0f}")


Total exposure (nilai transaksi bermasalah): Rp36,074,061,126
Kerugian terealisasi (bermasalah + Default): Rp5,366,212,184


**✍️ Insight & Rekomendasi untuk Tim Risk & Compliance:**

> 

Total exposure dari transaksi bermasalah (unauthorized + over-limit) mencapai Rp36,07 miliar. Dari jumlah 
ini, Rp5,37 miliar (dari 7.939 transaksi) sudah TEREALISASI menjadi kerugian karena berakhir Default — 
terbagi hampir merata antara user Basic tanpa hak PayLater (Rp2,69 miliar) dan user Plus yang melebihi 
limit (Rp2,68 miliar). Rekomendasi: (1) Suspend sementara fitur PayLater untuk mencegah exposure Rp30,7 
miliar sisanya (belum default) berubah jadi kerugian riil, sambil (2) sistem validasi diperbaiki agar 
mengecek eligibility tier DAN limit sebelum approve transaksi — bukan hanya salah satu.

#### 🔍 Investigasi — Angle 2: [Segmentasi User untuk Freeze vs Limit Adjustment]

**✍️ Mengapa kamu memilih angle ini?**

> 

Mengapa angle ini dipilih: Langsung menjawab poin (a) dan (b) — perlu segmentasi user, bukan blanket policy, supaya user yang masih sehat tidak ikut kena suspend.

In [61]:
# Angle 2
user_flags = golat.groupby('user_id').agg(
    n_trx=('amount','count'), n_violation=('flag', lambda x: (x!='valid').sum()),
    ever_default=('is_default','any'), total_amount=('amount','sum')
).reset_index()
user_flags['violation_rate'] = user_flags['n_violation']/user_flags['n_trx']

should_freeze = user_flags[(user_flags['n_violation']>0) & (user_flags['ever_default'])]
limit_adjust = user_flags[(user_flags['violation_rate']>0.5) & (~user_flags['ever_default'])]

print(f"Candidate FREEZE: {len(should_freeze)} user, total exposure Rp{should_freeze['total_amount'].sum():,.0f}")
print(f"Candidate LIMIT ADJUSTMENT: {len(limit_adjust)} user")

Candidate FREEZE: 8148 user, total exposure Rp18,748,282,753
Candidate LIMIT ADJUSTMENT: 9781 user


**✍️ Insight & Rekomendasi:**

> 

Dari 33.259 user unik yang pernah bertransaksi GoPayLater: 8.148 user (24,5%) memenuhi kriteria FREEZE 
segera (pernah melanggar limit DAN pernah Default) — membawa total exposure Rp18,75 miliar, atau lebih 
dari setengah total exposure bermasalah. Sementara itu 9.781 user (29,4%) tergolong candidate LIMIT 
ADJUSTMENT — sering melanggar limit (>50% transaksinya over-limit/unauthorized) tapi belum pernah 
Default, sehingga masih bisa diselamatkan dengan penyesuaian limit naik (jika memang mampu bayar) atau 
diperketat validasinya, tanpa perlu di-suspend penuh. Rekomendasi: prioritaskan investigasi manual pada 
8.148 user freeze-candidate terlebih dahulu karena kontribusinya paling besar terhadap risiko finansial.

---
### 4.5 Credit Risk Profiling *(Implicit — Open Ended)*

> Kamu diminta **Chief Risk Officer GoPay** untuk menyusun rekomendasi perbaikan algoritma credit scoring.  
> Tujuan: menentukan kriteria yang lebih ketat untuk pemberian limit PayLater,  
> sehingga NPL bisa ditekan tanpa terlalu banyak membatasi user yang sebenarnya *creditworthy*.

> 🧠 **Critical Thinking Prompt:**  
> Apakah user dengan credit score rendah **selalu** berisiko?  
> Bagaimana dengan user baru yang belum punya credit score sama sekali?  
> Temukan **sweet spot** antara risk mitigation dan business growth.

**Ekspektasi minimal:**
- Minimal 3 variabel/fitur berbeda yang kamu identifikasi sebagai prediktor default yang signifikan
- Profil 'high-risk user' berdasarkan kombinasi variabel tersebut
- Minimal 1 rekomendasi konkret untuk kebijakan limit PayLater yang berbasis data

**✍️ Definisi 'high-risk user' menurut kamu (dalam konteks kredit GoPay):**

> High-risk user adalah pengguna PayLater yang memiliki kombinasi indikator risiko, bukan hanya satu indikator tunggal. Indikator tersebut meliputi credit score rendah, penggunaan limit yang mendekati atau melampaui batas, serta frekuensi transaksi PayLater yang tinggi. User dengan credit score kosong tidak otomatis dikategorikan high-risk; user tersebut dapat diberikan limit awal yang kecil dan dievaluasi kembali berdasarkan perilaku transaksi serta riwayat pembayaran.

#### 📊 Prediktor Default 1: `credit_score_tier`

In [62]:
# Prediktor 1
# Menyiapkan dataframe analisis PayLater
df_risk = df_trx.copy()

# Merge profil user bila kolom user belum tersedia pada dataframe transaksi
if 'internal_credit_score' not in df_risk.columns:
    df_risk = df_risk.merge(
        df_users[['user_id', 'internal_credit_score', 'paylater_limit']],
        on='user_id',
        how='left'
    )

# Mengambil transaksi PayLater
payment_col = df_risk.get('payment_method_clean', df_risk['payment_method'])

df_paylater = df_risk[
    payment_col.astype(str)
    .str.lower()
    .str.replace('-', '', regex=False)
    .str.replace('_', '', regex=False)
    .eq('gopaylater')
].copy()

# Membentuk kelompok credit score, termasuk kategori Missing
df_paylater['credit_score_tier_analysis'] = pd.cut(
    df_paylater['internal_credit_score'],
    bins=[0, 599, 699, float('inf')],
    labels=['Low (<600)', 'Medium (600-699)', 'High (>=700)']
).astype('object')

df_paylater['credit_score_tier_analysis'] = (
    df_paylater['credit_score_tier_analysis']
    .fillna('Missing')
)

# Membandingkan default rate tiap kelompok
# Tambahkan flag default dari status pembayaran jika belum tersedia
if 'is_default' not in df_paylater.columns:
    df_paylater['is_default'] = (
        df_paylater['payment_status']
        .astype(str)
        .str.strip()
        .str.lower()
        .eq('default')
    )

predictor_1 = (
    df_paylater.groupby('credit_score_tier_analysis', observed=False)
    .agg(
        total_transaction=('trx_id', 'count'),
        default_rate=('is_default', 'mean')
    )
)

predictor_1['default_rate'] = (predictor_1['default_rate'] * 100).round(2)
display(predictor_1.sort_values('default_rate', ascending=False))

,total_transaction,default_rate
credit_score_tier_analysis,,
Medium (600-699),11851,15.16
Low (<600),59191,14.91
High (>=700),17966,14.79
Missing,15812,14.52


#### 📊 Prediktor Default 2: `paylater_usage_ratio`

In [63]:
# Prediktor 2
# Menghitung rasio penggunaan limit bila kolom belum tersedia
if 'paylater_usage_ratio' not in df_paylater.columns:
    df_paylater['paylater_usage_ratio'] = (
        df_paylater['amount'] / df_paylater['paylater_limit'].replace(0, np.nan)
    )

# Membuat kelompok intensitas penggunaan limit
df_paylater['usage_ratio_band'] = pd.cut(
    df_paylater['paylater_usage_ratio'],
    bins=[-float('inf'), 0.5, 0.8, 1.0, float('inf')],
    labels=['<50%', '50-80%', '80-100%', '>100%']
)

predictor_2 = (
    df_paylater.groupby('usage_ratio_band', observed=False)
    .agg(
        total_transaction=('trx_id', 'count'),
        default_rate=('is_default', 'mean')
    )
)

predictor_2['default_rate'] = (predictor_2['default_rate'] * 100).round(2)
display(predictor_2)


,total_transaction,default_rate
usage_ratio_band,,
<50%,42641,14.74
50-80%,5718,15.72
80-100%,3007,15.06
>100%,11831,14.83


#### 📊 Prediktor Default 3: `paylater_transaction_frequency`

In [64]:
# Prediktor 3
# Menghitung jumlah transaksi PayLater per user
user_frequency = (
    df_paylater.groupby('user_id')
    .agg(paylater_transaction_count=('trx_id', 'count'))
    .reset_index()
)

df_paylater = df_paylater.merge(user_frequency, on='user_id', how='left')

# Membuat kelompok frekuensi transaksi
# Pastikan kolom selalu tersedia meskipun cell dijalankan ulang
df_paylater['paylater_transaction_count'] = (
    df_paylater.groupby('user_id')['trx_id']
    .transform('count')
)

df_paylater['transaction_frequency_band'] = pd.cut(
    df_paylater['paylater_transaction_count'],
    bins=[0, 3, 7, float('inf')],
    labels=['1-3 transaksi', '4-7 transaksi', '8+ transaksi'],
    include_lowest=True
)

predictor_3 = (
    df_paylater.groupby('transaction_frequency_band', observed=False)
    .agg(
        total_transaction=('trx_id', 'count'),
        default_rate=('is_default', 'mean')
    )
)

predictor_3['default_rate'] = (predictor_3['default_rate'] * 100).round(2)
display(predictor_3)

,total_transaction,default_rate
transaction_frequency_band,,
1-3 transaksi,44308,14.91
4-7 transaksi,57067,14.86
8+ transaksi,3445,14.17


#### 🎯 Profil High-Risk User vs Average User

In [65]:
# Bandingkan karakteristik high-risk user vs keseluruhan user PayLater
# Membuat profil risiko di level user
user_risk_profile = (
    df_paylater.groupby('user_id')
    .agg(
        credit_score=('internal_credit_score', 'first'),
        max_usage_ratio=('paylater_usage_ratio', 'max'),
        paylater_transaction_count=('paylater_transaction_count', 'first'),
        default_rate=('is_default', 'mean')
    )
    .reset_index()
)

# User dengan score kosong tidak langsung dianggap high-risk
# Ia menjadi high-risk bila disertai penggunaan limit tinggi atau frekuensi tinggi.
frequency_threshold = user_risk_profile['paylater_transaction_count'].quantile(0.75)

user_risk_profile['low_score_flag'] = (
    user_risk_profile['credit_score'].lt(600)
)

user_risk_profile['high_usage_flag'] = (
    user_risk_profile['max_usage_ratio'].ge(0.80)
)

user_risk_profile['high_frequency_flag'] = (
    user_risk_profile['paylater_transaction_count'].ge(frequency_threshold)
)

user_risk_profile['risk_point'] = (
    user_risk_profile['low_score_flag'].astype(int)
    + user_risk_profile['high_usage_flag'].astype(int)
    + user_risk_profile['high_frequency_flag'].astype(int)
)

# High risk bila memiliki minimal dua indikator
user_risk_profile['risk_profile'] = np.where(
    user_risk_profile['risk_point'] >= 2,
    'High Risk',
    'Average / Lower Risk'
)

risk_comparison = (
    user_risk_profile.groupby('risk_profile')
    .agg(
        total_user=('user_id', 'nunique'),
        avg_credit_score=('credit_score', 'mean'),
        avg_max_usage_ratio=('max_usage_ratio', 'mean'),
        avg_transaction_frequency=('paylater_transaction_count', 'mean'),
        avg_default_rate=('default_rate', 'mean')
    )
)

risk_comparison['avg_default_rate'] = (
    risk_comparison['avg_default_rate'] * 100
).round(2)

display(risk_comparison)

,total_user,avg_credit_score,avg_max_usage_ratio,avg_transaction_frequency,avg_default_rate
risk_profile,,,,,
Average / Lower Risk,22330,555.09,0.65,2.50,14.80
High Risk,10929,498.04,1.86,4.49,15.00


**✍️ Rekomendasi Kebijakan Limit PayLater untuk Chief Risk Officer:**

> Terapkan kebijakan limit bertahap (progressive limit). User dengan credit score rendah tidak harus langsung ditolak, tetapi diberi limit awal yang lebih kecil. User baru dengan credit score kosong juga dapat memperoleh limit konservatif setelah verifikasi identitas. Kenaikan limit hanya diberikan apabila user memiliki riwayat pembayaran baik, penggunaan limit yang sehat, dan tidak menunjukkan frekuensi transaksi yang berisiko. Sebaliknya, user dengan kombinasi minimal dua indikator risiko perlu menerima penurunan limit, verifikasi tambahan, atau rekomendasi Freeze Account sementara.

---
## 5. Export Clean Dataset

In [66]:
# Gabungkan ketiga tabel menjadi satu dataframe final
# Gunakan LEFT JOIN dengan df_trx sebagai tabel utama
# Sertakan semua fitur baru yang telah dibuat

# TODO: Implementasi JOIN
# df_final = df_trx.merge(df_users[...], on='user_id', how='left')
#                  .merge(df_services[...], on='service_id', how='left')

# Export
# df_final.to_csv('gopay_clean.csv', index=False)
# print(f'Dataset berhasil disimpan: gopay_clean.csv')
# print(f'Shape final: {df_final.shape}')
# print(f'Kolom baru yang ditambahkan: {[c for c in df_final.columns if c not in df_trx_raw.columns]}')

# Simpan daftar kolom transaksi awal untuk melihat kolom hasil feature engineering
original_trx_columns = df_trx_raw.columns.tolist()

# Pilih kolom profil user yang belum ada pada df_trx
user_columns_to_add = [
    col for col in df_users.columns
    if col != 'user_id' and col not in df_trx.columns
]

# Pilih kolom layanan yang belum ada pada df_trx
service_columns_to_add = [
    col for col in df_services.columns
    if col != 'service_id' and col not in df_trx.columns
]

# LEFT JOIN: pertahankan seluruh transaksi sebagai tabel utama
df_final = (
    df_trx
    .merge(
        df_users[['user_id'] + user_columns_to_add],
        on='user_id',
        how='left'
    )
    .merge(
        df_services[['service_id'] + service_columns_to_add],
        on='service_id',
        how='left'
    )
)

# Validasi jumlah baris: LEFT JOIN tidak boleh mengubah jumlah transaksi
assert len(df_final) == len(df_trx), "Jumlah baris berubah setelah proses merge."

# Export
output_file = 'gopay_clean.csv'
df_final.to_csv(output_file, index=False)

new_columns = [col for col in df_final.columns if col not in original_trx_columns]

print(f"Dataset berhasil disimpan: {output_file}")
print(f"Shape final: {df_final.shape}")
print(f"Jumlah transaksi: {len(df_final):,}")
print(f"Kolom baru yang ditambahkan: {new_columns}")

display(df_final.head())

Dataset berhasil disimpan: gopay_clean.csv
Shape final: (300000, 21)
Jumlah transaksi: 300,000
Kolom baru yang ditambahkan: ['internal_credit_score', 'is_high_risk_transaction', 'join_date', 'gopay_tier', 'user_tenure_days', 'category']


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,is_duplicate_trxid,unknown_service,inconsistent_status,internal_credit_score,payment_method_clean,paylater_limit,is_paylater_violation,is_high_risk_transaction,join_date,gopay_tier,user_tenure_days,service_name,category
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid,False,False,False,349.00,gopay,0,False,False,NaT,NaN,NaN,Investasi,Digital Goods & Bills
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid,False,False,False,589.00,gopay,3000000,False,False,2022-05-02,Plus,608.00,Investasi,Digital Goods & Bills
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,gopay,34229,0.00,Paid,False,False,False,726.00,gopay,500000,False,False,2021-12-06,Plus,755.00,GoRide,Mobility
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,gopay,234183,0.00,Paid,False,False,False,349.00,gopay,0,False,False,NaT,NaN,NaN,GoRide,Mobility
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,PayLater,380431,0.00,Paid,False,False,False,553.00,gopaylater,1500000,False,False,2022-03-05,Plus,666.00,GoCar,Mobility


---
## 6. Ringkasan & Refleksi

**Keputusan Data Cleaning yang paling challenging dan mengapa:**

> 

**Temuan paling menarik dari EDA (khususnya terkait risiko kredit):**

> 

**Rekomendasi bisnis utama yang bisa diberikan kepada tim Risk Management GoPay:**

> 

# Keputusan Data Cleaning yang paling challenging dan mengapa:
Keputusan paling menantang adalah penanganan transaksi GoPayLater yang melebihi limit dan transaksi user Basic dengan limit Rp0. Data tersebut tidak dihapus karena bukan sekadar data kotor, melainkan bukti adanya business logic error pada proses otorisasi PayLater. Transaksi tetap dipertahankan dan diberi flag agar dampak risiko kredit serta potensi kerugian dapat dianalisis secara terpisah.

# Temuan paling menarik dari EDA khususnya terkait risiko kredit:
Ditemukan 53.454 transaksi GoPayLater yang melebihi limit, atau sekitar 51% dari seluruh transaksi GoPayLater. Sebanyak 41.623 transaksi berasal dari user Basic yang seharusnya tidak memiliki akses karena limitnya Rp0. Selain itu, terdapat 5.250 user tanpa internal credit score, 75 nilai late fee negatif, dan 30.177 transaksi yang tercatat sebelum tanggal user bergabung. Temuan ini menunjukkan bahwa peningkatan risiko kredit tidak hanya berasal dari perilaku user, tetapi juga dari masalah kualitas data dan kontrol sistem.

# Rekomendasi bisnis utama yang bisa diberikan kepada tim Risk Management GoPay:
Prioritas utama adalah menerapkan validasi limit dan eligibility check secara real-time pada sisi server sebelum transaksi GoPayLater disetujui. User Basic atau user dengan limit Rp0 harus otomatis ditolak, sedangkan transaksi yang melebihi sisa limit harus tidak dapat diproses. Sebagai pelengkap, GoPay dapat menerapkan fitur Freeze Account untuk memberi pengguna cara cepat menghentikan transaksi PayLater ketika terdapat indikasi penyalahgunaan akun. Untuk user baru atau user dengan credit score kosong, gunakan limit awal konservatif dan naikkan secara bertahap berdasarkan riwayat pembayaran serta perilaku penggunaan limit.